# Import & GPU Utils

In [2]:
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

import torch
import random
import os

import torch.nn as nn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

USE_AMP = (DEVICE.type == "cuda" and os.environ.get("TSF_USE_AMP", "1") != "0")
USE_TORCH_COMPILE = (os.environ.get("TSF_TORCH_COMPILE", "0") == "1")

In [3]:
TARGET_COL = "max_temperature"
FEATURE_COLS = [
    "max_temperature",
    "latitude",
    "longitude",
    "elevation",
    "sin_doy",
    "cos_doy",
    "solar_declination",
    "dmi east",
    "nino anom 3.4",
]
INPUT_LEN = 14
HORIZON = 7
STRIDE = 2


LOCAL_TEMPORAL_COLS = ["max_temperature", "sin_doy", "cos_doy", "solar_declination"]
GEO_COLS = ["latitude", "longitude", "elevation"]
CLIMATE_COLS = ["nino anom 3.4", "dmi east"]


In [4]:
_CPU_CORES = os.cpu_count() or 4
# Threadripper 3960X = 24 cores (48 threads). For NumPy/torch this is a good default.
DEFAULT_CPU_THREADS = min(24, _CPU_CORES)

# Only set if user hasn't already configured these in their environment.
os.environ.setdefault("OMP_NUM_THREADS", str(DEFAULT_CPU_THREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(DEFAULT_CPU_THREADS))


import random
import torch



torch.set_default_dtype(torch.float32)
torch.set_num_threads(DEFAULT_CPU_THREADS)
# torch.set_num_interop_threads(max(1, min(4, DEFAULT_CPU_THREADS // 2)))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DEVICE.type == "cuda":
    # Ampere GPU (3090 Ti): TF32 is a big speedup for matmul/conv with minimal impact for this task.
    print(f"Training With: {DEVICE}")
    _use_tf32 = (os.environ.get("TSF_USE_TF32", "1") != "0")
    if _use_tf32:
        try:
            torch.backends.cuda.matmul.allow_tf32 = True
            torch.backends.cudnn.allow_tf32 = True
        except Exception:
            pass
    torch.backends.cudnn.benchmark = True

USE_AMP = (DEVICE.type == "cuda" and os.environ.get("TSF_USE_AMP", "1") != "0")
USE_TORCH_COMPILE = (os.environ.get("TSF_TORCH_COMPILE", "0") == "1")


def _maybe_compile(model: torch.nn.Module) -> torch.nn.Module:
    if not USE_TORCH_COMPILE:
        return model
    try:
        return torch.compile(model)  # PyTorch 2.x
    except Exception:
        return model


def _dataloader_kwargs(shuffle: bool, drop_last: bool = False):
    """
    Sensible DataLoader defaults for this workstation.
    - Use CPU workers to overlap preprocessing/host->device transfers.
    - Use pin_memory for faster H2D copies when using CUDA.
    """
    num_workers = 0
    if DEVICE.type == "cuda":
        # Windows multiprocessing can be heavier; keep this conservative but non-zero.
        num_workers = min(8, max(2, DEFAULT_CPU_THREADS // 3))

    kwargs = dict(
        shuffle=bool(shuffle),
        num_workers=int(num_workers),
        pin_memory=(DEVICE.type == "cuda"),
        drop_last=bool(drop_last),
    )
    if kwargs["num_workers"] > 0:
        kwargs["persistent_workers"] = True
        kwargs["prefetch_factor"] = 2
    return kwargs


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

Training With: cuda


In [5]:
results_rows = []
def log_row(**kwargs):
    row = {
            "target_region": "papua",
            "source_region": "java",
            "TARGET_COL": TARGET_COL,
            "INPUT_LEN": INPUT_LEN,
            "HORIZON": HORIZON,
            "STRIDE": STRIDE,
            "n_features": len(FEATURE_COLS),
        }
    row.update(kwargs)
    results_rows.append(row)

# Windowing

In [6]:
def load_and_split(data_path: str):
    """Load merged_dataset.csv and split by time. Return (train, val, test) DataFrames."""
    df = pd.read_csv(data_path, dtype={"location_id": str})
    # Normalize column names (e.g. if CSV uses "temperature_2m_max (°C)" or "DMI EAST")
    col_map = {}
    for c in list(df.columns):
        c_lower = c.strip().lower()
        if "temperature" in c_lower and "max" in c_lower and c != "max_temperature":
            col_map[c] = "max_temperature"
        if "dmi" in c_lower and "east" in c_lower and c != "dmi east":
            col_map[c] = "dmi east"
        if "nino" in c_lower and "3.4" in c_lower and c != "nino anom 3.4":
            col_map[c] = "nino anom 3.4"
    df = df.rename(columns=col_map)
    df["time"] = pd.to_datetime(df["time"])
    df = df.sort_values(["location_id", "time"]).reset_index(drop=True)

    train = df[(df["time"] >= "2005-01-01") & (df["time"] <= "2018-12-31")]
    val   = df[(df["time"] >= "2019-01-01") & (df["time"] <= "2021-12-31")]
    test  = df[(df["time"] >= "2022-01-01") & (df["time"] <= "2025-05-01")]

    return train, val, test


def build_windows_one_location(times, values, input_len, horizon, stride):
    """Build (X, y) windows for one location. values: (T, n_features)."""
    T = len(times)
    # if the station does not have enough values to form a full window # we skip
    if T < input_len + horizon:
        return None, None
    X_list, y_list = [], []
    for start in range(0, T - input_len - horizon + 1, stride):
        end_in = start + input_len
        end_out = end_in + horizon
        X_list.append(values[start:end_in])
        y_list.append(values[end_in:end_out, 0])
    if not X_list:
        return None, None
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)


def build_windows_temp_transformer_one_location(times, values, input_len, horizon, stride):
    """Build windows for `TemperatureTransformer`.

    Returns:
      - z_past: (input_len, 1) temperature history
      - x_cov_past: (input_len, d_cov) past covariates (features 1:)
      - x_cov_future: (horizon, d_cov) future covariates (features 1:)
      - y: (horizon,) future temperature target (feature 0)

    `values` must be shaped (T, n_features) where feature 0 is temperature.
    """
    T = len(times)
    if T < input_len + horizon:
        return None, None, None, None

    z_past_list, x_cov_past_list, x_cov_future_list, y_list = [], [], [], []

    for start in range(0, T - input_len - horizon + 1, stride):
        end_in = start + input_len
        end_out = end_in + horizon

        past = values[start:end_in]       # (input_len, n_features)
        future = values[end_in:end_out]  # (horizon, n_features)

        z_past_list.append(past[:, :1])
        x_cov_past_list.append(past[:, 1:])
        x_cov_future_list.append(future[:, 1:])
        y_list.append(future[:, 0])

    if not z_past_list:
        return None, None, None, None

    return (
        np.array(z_past_list, dtype=np.float32),
        np.array(x_cov_past_list, dtype=np.float32),
        np.array(x_cov_future_list, dtype=np.float32),
        np.array(y_list, dtype=np.float32),
    )


def build_all_windows(train, val, test):
    """Build X, y for train/val/test. No cross location_id, no cross split boundary."""
    def run_split(split_df):
        X_list, y_list = [],[]
        grp = grp.sort_values("time")
        mat = grp[FEATURE_COLS].values.astype(np.float32)
        times = grp["time"].values
        X, y = build_windows_one_location(times, mat, INPUT_LEN, HORIZON, STRIDE)
        if not X is not None:
            X_list.append(X)
            y_list.append(y)
        if not X_list:
            return np.zeros((0, INPUT_LEN, len(FEATURE_COLS)), dtype=np.float32), np.zeros((0, INPUT_LEN, len(FEATURE_COLS)), dtype=np.float32)
        return np.concatenate(X_list, axis=0), np.concatenate(y_list, axis=0)
    X_train, y_train = run_split(train)
    X_val,   y_val   = run_split(val)
    X_test,  y_test  = run_split(test)
    return X_train, y_train, X_val, y_val, X_test, y_test




In [7]:
def build_windows(split_df: pd.DataFrame):
    """Build(X, y) for a single split dataframe (across all location_id)"""
    X_list, y_list = [], []
    for _, grp in split_df.groupby("location_id"):
        grp = grp.sort_values("time")
        mat = grp[FEATURE_COLS].values.astype(np.float32)
        times = grp['time'].values
        X, y = build_windows_one_location(times, mat, INPUT_LEN, HORIZON, STRIDE)
        if X is not None:
            X_list.append(X)
            y_list.append(y)
    if not X_list:
        return (
            np.zeros((0, INPUT_LEN, len(FEATURE_COLS)), dtype=np.float32),
            np.zeros((0, HORIZON), dtype=np.float32)
        )
    return np.concatenate(X_list, axis=0), np.concatenate(y_list, axis=0)


def build_windows_temp_transformer(split_df: pd.DataFrame):
    """Build windows for `TemperatureTransformer` across all `location_id`.

    Returns:
      - z_past: (N, INPUT_LEN, 1)
      - x_cov_past: (N, INPUT_LEN, d_cov) where d_cov=len(FEATURE_COLS)-1
      - x_cov_future: (N, HORIZON, d_cov)
      - y: (N, HORIZON) future temperature target
    """
    z_list, x_cov_past_list, x_cov_future_list, y_list = [], [], [], []

    d_cov = len(FEATURE_COLS) - 1

    for _, grp in split_df.groupby("location_id"):
        grp = grp.sort_values("time")
        mat = grp[FEATURE_COLS].values.astype(np.float32)
        times = grp['time'].values

        z_past, x_cov_past, x_cov_future, y = build_windows_temp_transformer_one_location(
            times, mat, INPUT_LEN, HORIZON, STRIDE
        )

        if z_past is not None:
            z_list.append(z_past)
            x_cov_past_list.append(x_cov_past)
            x_cov_future_list.append(x_cov_future)
            y_list.append(y)

    if not z_list:
        return (
            np.zeros((0, INPUT_LEN, 1), dtype=np.float32),
            np.zeros((0, INPUT_LEN, d_cov), dtype=np.float32),
            np.zeros((0, HORIZON, d_cov), dtype=np.float32),
            np.zeros((0, HORIZON), dtype=np.float32),
        )

    return (
        np.concatenate(z_list, axis=0),
        np.concatenate(x_cov_past_list, axis=0),
        np.concatenate(x_cov_future_list, axis=0),
        np.concatenate(y_list, axis=0),
    )

In [8]:
## Create a boolean fileter to keep only the true rows getting returned
def region_mask(df:pd.DataFrame, region_name: str):
    return df["region"].astype(str).str.strip().str.lower() == region_name.strip().lower()

# Filter Station

In [9]:
def compute_station_stats(papua_train: pd.DataFrame) -> pd.DataFrame:
    if papua_train is None or len(papua_train) == 0:
        return pd.DataFrame(columns=["location_id", "mean_temp", "std_temp", "elevation", "n_rows"])
    df = papua_train.copy()
    df['location_id'] = df['location_id'].astype(str)
    g = df.groupby("location_id", dropna=False)
    
    stats = g.agg(
        mean_temp=(TARGET_COL, "mean"),
        std_temp=(TARGET_COL, "std"),
        elevation=("elevation", "mean"),
        n_rows=(TARGET_COL, "size"),
    ).reset_index()
    
    stats["mean_temp"] = stats["mean_temp"].astype(np.float32)
    stats["std_temp"] = stats["std_temp"].fillna(0.0).astype(np.float32)
    stats["elevation"] = stats["elevation"].astype(np.float32)
    stats["n_rows"] = stats["n_rows"].astype(int)
    return stats

def select_target_stations_papua_by_elevation(papua_train: pd.DataFrame):
    """
    Select station IDs from Papua training set by elevation:
      - Single station: median elevation station
      - Three stations: lowest, median, highest elevation stations
    Returns (single_station_id: str, three_station_ids: list[str], stats_sorted: pd.DataFrame).
    """
    stats = compute_station_stats(papua_train)
    if len(stats) == 0:
        return None, [], stats

    stats_sorted = stats.sort_values(["elevation", "location_id"], ascending=[True, True]).reset_index(drop=True)
    mid = len(stats_sorted) // 2
    single_id = str(stats_sorted.loc[mid, "location_id"])

    low_id = str(stats_sorted.loc[0, "location_id"])
    high_id = str(stats_sorted.loc[len(stats_sorted) - 1, "location_id"])
    three_ids = [low_id, single_id, high_id]

    # If very small number of stations, deduplicate while keeping order
    seen = set()
    three_ids = [x for x in three_ids if not (x in seen or seen.add(x))]
    return single_id, three_ids, stats_sorted


def filter_df_by_station_ids(df: pd.DataFrame, station_ids) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return df.iloc[0:0].copy() if df is not None else df
    ids = set([str(x) for x in station_ids])
    out = df.copy()
    out["location_id"] = out["location_id"].astype(str)
    return out[out["location_id"].isin(ids)]

# Model Architecuture

## Vanilla Transformer

### Architecture

In [9]:
import torch.nn as nn

class VanillaTransformer(nn.Module):
    def __init__(self, input_dim=9, d_model=32, nhead=4, num_layers=2, dropout=0.1, horizon=7):
        """
        Model Parameter:
          - input_dim = 9. Number of feature
          - d_model = 32. Internal Feature Size of transformer. 9 Features -> projected to 32 features
          - nhead = 4. Number of Attention Heads
          - num_layers = 2.  Stack 2 Transformer Layers. Each Layer has (self-attention + Feedforward Network)
          - Horizon = 7. Predict 7 days ahead
        """

        super().__init__()
        ## input Projection. Transform from 9 features to 32 Features
        self.input_proj = nn.Linear(input_dim, d_model)
        # transformer decoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*4,
            dropout=dropout, batch_first=True, norm_first=False
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, horizon)

    def forward(self, x):
        # x: (B,T,F)
        x = self.input_proj(x)
        x = self.encoder(x)
        x = x.mean(dim=1)
        return self.head(x)

### Evals Vanilla Transformer (Supervised)

In [21]:
def eval_model_mae(model: torch.nn.Module, X: np.ndarray, y: np.ndarray, batch_size: int = 256) -> float:
    if X.shape[0] == 0:
        return float("nan")
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, X.shape[0], batch_size):
            xb = torch.from_numpy(X[i : i + batch_size])
            xb = xb.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
            preds.append(model(xb).detach().cpu().numpy().astype(np.float32))
    pred = np.concatenate(preds, axis=0)
    return float(np.mean(np.abs(pred - y)))

In [22]:
def eval_model_metrics(model: torch.nn.Module, X: np.ndarray, y: np.ndarray, batch_size: int = 256):
    """
    Compute MAE/MSE/RMSE on window targets (y shape: [N, H]).
    Metrics are computed over all elements (N*H).
    Returns dict: {"mae": float, "mse": float, "rmse": float}.
    """
    if X.shape[0] == 0:
        return {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")}
    model.eval() # switch model into evaluation mode
    preds = []
    with torch.no_grad(): # disable gradients
        for i in range(0, X.shape[0], batch_size):
            # Extract batch 
            xb = torch.from_numpy(X[i : i + batch_size])
            xb = xb.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
            preds.append(model(xb).detach().cpu().numpy().astype(np.float32))
    pred = np.concatenate(preds, axis=0).astype(np.float32)
    y = y.astype(np.float32, copy=False)
    err = (pred - y).astype(np.float32)
    mse = float(np.mean(err ** 2))
    mae = float(np.mean(np.abs(err)))
    rmse = float(np.sqrt(mse))
    return {"mae": mae, "mse": mse, "rmse": rmse}



### Train Vanilla Transformer (Supervised)

In [12]:
def train_supervised_earlystop_target_mae(
    model: nn.Module,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_tgt_val: np.ndarray,
    y_tgt_val: np.ndarray,
    epochs: int = 100,
    batch_size: int = 256,
    lr: float = 1e-3,
    patience: int = 5,
):
    """Train with MSE; early stop on target validation MAE."""
    from torch.utils.data import TensorDataset, DataLoader

    if X_train.shape[0] == 0:
        return model.to(DEVICE), float("nan")

    model = model.to(DEVICE).float()
    train_ds = TensorDataset(torch.from_numpy(X_train).to(DEVICE), torch.from_numpy(y_train).to(DEVICE))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    best_mae, best_state, wait = float("inf"), None, 0

    for ep in range(epochs):
        model.train()
        tr_mse = 0.0
        for xb, yb in train_loader:
            opt.zero_grad() # resetting the gradient from the previous batch. But why?? -> so on the next training iteration the gradient is get set to 0
            pred = model(xb)
            loss = nn.functional.mse_loss(pred, yb)
            loss.backward() # back propagation
            opt.step() # update weights
            tr_mse += loss.item() * xb.size(0) # *** track training loss (accumulates loss accross batches)
        tr_mse /= max(1, X_train.shape[0]) # *** mean training loss per sample

        val_mae = eval_model_mae(model, X_tgt_val, y_tgt_val, batch_size=256)
        print(f"  Epoch {ep+1}/{epochs}  Train MSE: {tr_mse:.6f}  Target Val MAE: {val_mae:.4f}")

        if val_mae < best_mae:
            best_mae = val_mae
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        # else:
        #     wait += 1
        #     if wait >= patience:
        #         print(f"  Early stopping at epoch {ep+1}")
        #         break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model.to(DEVICE), best_mae


## Vanilla Transformer Domain Adversarial NN

### Feature Extractor

In [13]:
class FeatureExtractor(nn.Module):
    """Transformer encoder + mean pooling. Output: pooled representation (d_model)."""

    def __init__(self, input_dim: int, d_model: int = 32, nhead: int = 4, num_layers: int = 2, dropout: float = 0.1):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=False,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.input_proj(x)
        x = self.encoder(x)
        return x.mean(dim=1)

### Domain Classifier

In [2]:
import torch.nn as nn
import torch

class DomainClassifier(nn.Module):
    """MLP: Linear(in→32) ReLU Linear(32→1) Sigmoid."""

    def __init__(self, in_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(1)  # (B,)

def dann_lambda_schedule(p: float) ->float:
    import math
    return float(2.0/ (1.0 + math.exp(-10.0 * float(p))) - 1.0)



### Gradient Reversal Layer

In [15]:
class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x: torch.Tensor, lambd: float):
        ctx.lambd = float(lambd)
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output: torch.Tensor):
        return -ctx.lambd * grad_output, None


def grl(x: torch.Tensor, lambd: float) -> torch.Tensor:
    return GradReverse.apply(x, lambd)

### Train DANN

In [16]:
def train_domain_adversarial_dann(
    feat_extractor: nn.Module,
    task_head: nn.Module,
    domain_clf: nn.Module,
    X_src: np.ndarray,
    y_src: np.ndarray,
    X_tgt: np.ndarray,
    y_tgt: np.ndarray,
    X_tgt_val: np.ndarray,
    y_tgt_val: np.ndarray,
    epochs: int = 100,
    batch_size: int = 256,
    lr: float = 1e-3,
    patience: int = 5,
    use_target_task_loss: bool = False,
):
    """
    DANN:
      loss = forecast_MSE + λ * domain_BCE
      λ schedule: 2/(1+exp(-10p)) - 1, p=step/total_steps
      balanced batches: batch_size/2 source + batch_size/2 target
      early stop: target validation MAE
    """
    from torch.utils.data import TensorDataset, DataLoader

    if X_src.shape[0] == 0 or X_tgt.shape[0] == 0:
        return None, float("nan")

    src_bs = max(1, batch_size // 2)
    tgt_bs = max(1, batch_size // 2)

    src_ds = TensorDataset(torch.from_numpy(X_src).to(DEVICE), torch.from_numpy(y_src).to(DEVICE))
    tgt_ds = TensorDataset(torch.from_numpy(X_tgt).to(DEVICE), torch.from_numpy(y_tgt).to(DEVICE))
    src_loader = DataLoader(src_ds, batch_size=src_bs, shuffle=True, num_workers=0, drop_last=True)
    tgt_loader = DataLoader(tgt_ds, batch_size=tgt_bs, shuffle=True, num_workers=0, drop_last=True)

    feat_extractor = feat_extractor.to(DEVICE).float()
    task_head = task_head.to(DEVICE).float()
    domain_clf = domain_clf.to(DEVICE).float()

    params = list(feat_extractor.parameters()) + list(task_head.parameters()) + list(domain_clf.parameters())
    opt = torch.optim.Adam(params, lr=lr)
    bce = nn.BCELoss()

    total_steps = epochs * min(len(src_loader), len(tgt_loader))
    global_step = 0

    best_mae, best_state, wait = float("inf"), None, 0

    for ep in range(epochs):
        feat_extractor.train()
        task_head.train()
        domain_clf.train()

        tr_task, tr_dom = 0.0, 0.0
        n_steps = min(len(src_loader), len(tgt_loader))
        src_iter = iter(src_loader)
        tgt_iter = iter(tgt_loader)

        for _ in range(n_steps):
            xs, ys = next(src_iter)
            xt, yt = next(tgt_iter)
            xb = torch.cat([xs, xt], dim=0)
            dom_y = torch.cat(
                [torch.zeros(xs.size(0), device=DEVICE), torch.ones(xt.size(0), device=DEVICE)],
                dim=0,
            ).float()

            p = global_step / max(1, total_steps)
            lambd = dann_lambda_schedule(p)
            global_step += 1

            opt.zero_grad()
            rep = feat_extractor(xb)
            yhat = task_head(rep)

            task_loss = nn.functional.mse_loss(yhat[: xs.size(0)], ys)
            if use_target_task_loss:
                task_loss = task_loss + nn.functional.mse_loss(yhat[xs.size(0) :], yt)

            dom_pred = domain_clf(grl(rep, lambd))
            dom_loss = bce(dom_pred, dom_y)

            loss = task_loss + (lambd * dom_loss)
            loss.backward()
            opt.step()

            tr_task += float(task_loss.item())
            tr_dom += float(dom_loss.item())

        eval_model = nn.Sequential(feat_extractor, task_head)
        val_mae = eval_model_mae(eval_model, X_tgt_val, y_tgt_val, batch_size=256)
        tr_task /= max(1, n_steps)
        tr_dom /= max(1, n_steps)
        print(f"  Epoch {ep+1}/{epochs}  Task MSE: {tr_task:.6f}  Domain BCE: {tr_dom:.6f}  Target Val MAE: {val_mae:.4f}")

        if val_mae < best_mae:
            best_mae = val_mae
            best_state = {
                "feat": {k: v.cpu().clone() for k, v in feat_extractor.state_dict().items()},
                "task": {k: v.cpu().clone() for k, v in task_head.state_dict().items()},
                "dom": {k: v.cpu().clone() for k, v in domain_clf.state_dict().items()},
            }
        #     wait = 0
        # else:
        #     wait += 1
        #     if wait >= patience:
        #         print(f"  Early stopping at epoch {ep+1}")
        #         break

    if best_state is not None:
        feat_extractor.load_state_dict(best_state["feat"])
        task_head.load_state_dict(best_state["task"])
        domain_clf.load_state_dict(best_state["dom"])

    return nn.Sequential(feat_extractor, task_head).to(DEVICE), best_mae



## Climate Aware Vanilla Transformer Based

### Non-DA

In [17]:
class ClimateAwareTransformer(nn.Module):
    """
    Temporal local features -> Transformer encoder -> 32-dim pooled
    Geo features (lat, lon, elev) -> Linear(3->16)
    Climate regime (Nino, DMI) -> Linear(2->16)
    Concat -> 64 -> Linear(64->7)
    """

    def __init__(self, horizon: int = 7, d_model: int = 32, nhead: int = 4, num_layers: int = 2, dropout: float = 0.1):
        super().__init__()
        self.idx_temp = [FEATURE_COLS.index(c) for c in LOCAL_TEMPORAL_COLS]
        self.idx_geo = [FEATURE_COLS.index(c) for c in GEO_COLS]
        self.idx_clim = [FEATURE_COLS.index(c) for c in CLIMATE_COLS]

        self.temporal = FeatureExtractor(input_dim=len(self.idx_temp), d_model=d_model, nhead=nhead, num_layers=num_layers, dropout=dropout)
        self.geo_proj = nn.Linear(3, 16)
        self.clim_proj = nn.Linear(2, 16)
        self.head = nn.Linear(d_model + 16 + 16, horizon)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        xt = x[:, :, self.idx_temp]
        rep_t = self.temporal(xt)
        rep_g = self.geo_proj(x[:, 0, self.idx_geo])
        rep_c = self.clim_proj(x[:, 0, self.idx_clim])
        rep = torch.cat([rep_t, rep_g, rep_c], dim=1)
        return self.head(rep)

### DA

In [18]:
class ClimateAwareRep(nn.Module):
    """Same as ClimateAwareTransformer but returns 64-dim representation (for optional DANN)."""

    def __init__(self, d_model: int = 32, nhead: int = 4, num_layers: int = 2, dropout: float = 0.1):
        super().__init__()
        self.idx_temp = [FEATURE_COLS.index(c) for c in LOCAL_TEMPORAL_COLS]
        self.idx_geo = [FEATURE_COLS.index(c) for c in GEO_COLS]
        self.idx_clim = [FEATURE_COLS.index(c) for c in CLIMATE_COLS]

        self.temporal = FeatureExtractor(input_dim=len(self.idx_temp), d_model=d_model, nhead=nhead, num_layers=num_layers, dropout=dropout)
        self.geo_proj = nn.Linear(3, 16)
        self.clim_proj = nn.Linear(2, 16)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        xt = x[:, :, self.idx_temp]
        rep_t = self.temporal(xt)
        rep_g = self.geo_proj(x[:, 0, self.idx_geo])
        rep_c = self.clim_proj(x[:, 0, self.idx_clim])
        return torch.cat([rep_t, rep_g, rep_c], dim=1)  # (B, 64)

## TFT Non DA

### Gated Residual Network

In [16]:
class GRN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim=None, dropout=0.1):
        super().__init__()
        output_dim = output_dim or input_dim

        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.elu = nn.ELU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)

        self.dropout = nn.Dropout(dropout)
        self.gate = nn.Linear(output_dim, output_dim)
        self.norm = nn.LayerNorm(output_dim)

        if input_dim != output_dim:
            self.skip = nn.Linear(input_dim, output_dim)
        else:
            self.skip = None

    def forward(self, x):
        residual = x if self.skip is None else self.skip(x)

        out = self.fc1(x)
        out = self.elu(out)
        out = self.fc2(out)
        out = self.dropout(out)

        gate = torch.sigmoid(self.gate(out))
        out = gate * out + (1 - gate) * residual

        return self.norm(out)

### Variable Selection Network

In [17]:
class VariableSelection(nn.Module):
    def __init__(self, num_vars, d_model):
        super().__init__()
        self.weight_net = nn.Sequential(
            nn.Linear(num_vars, num_vars),
            nn.Softmax(dim=-1)
        )
        self.proj = nn.Linear(num_vars, d_model)

    def forward(self, x):
        # x: (B, T, num_vars)
        weights = self.weight_net(x.mean(dim=1))  # (B, num_vars)
        x_weighted = x * weights.unsqueeze(1)
        return self.proj(x_weighted)

### Full Model

In [18]:
class ClimateTFT(nn.Module):
    def __init__(
        self,
        horizon=7,
        d_model=32,
        nhead=4,
        num_layers=2,
        dropout=0.1,
    ):
        super().__init__()

        self.idx_temp = [FEATURE_COLS.index(c) for c in LOCAL_TEMPORAL_COLS]
        self.idx_geo = [FEATURE_COLS.index(c) for c in GEO_COLS]
        self.idx_clim = [FEATURE_COLS.index(c) for c in CLIMATE_COLS]

        # --- Variable Selection ---f
        self.vsn = VariableSelection(len(self.idx_temp), d_model)

        # --- Transformer (same as before but cleaner) ---
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # --- Static encoders (GRN instead of Linear) ---
        self.geo_grn = GRN(3, 32, 16)
        self.clim_grn = GRN(2, 32, 16)

        # --- Temporal GRN ---
        self.temporal_grn = GRN(d_model, 64, d_model)

        # --- Fusion GRN ---
        self.fusion_grn = GRN(d_model + 16 + 16, 64, 64)

        # --- Output ---
        self.head = nn.Linear(64, horizon)

    def forward(self, x):
        # Split inputs
        xt = x[:, :, self.idx_temp]
        xg = x[:, 0, self.idx_geo]
        xc = x[:, 0, self.idx_clim]

        # --- Variable selection ---
        xt = self.vsn(xt)  # (B, T, d_model)

        # --- Transformer ---
        h = self.transformer(xt)

        # Pooling (same as before)
        h = h.mean(dim=1)

        # --- Temporal refinement ---
        h = self.temporal_grn(h)

        # --- Static features ---
        g = self.geo_grn(xg)
        c = self.clim_grn(xc)

        # --- Fusion ---
        rep = torch.cat([h, g, c], dim=1)
        rep = self.fusion_grn(rep)

        return self.head(rep)

In [19]:
def train_tft(
    model: nn.Module,
    X_train: pd.DataFrame,
    y_train: pd.DataFrame,
    X_val: pd.DataFrame,
    y_val: pd.DataFrame,
    epochs=100,
    batch_size=256,
    lr=1e-3,
    patience=5
):
    model = model.to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    best_val = float("inf")
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        perm = np.random.permutation(len(X_train))

        for i in range(0, len(X_train), batch_size):
            idx = perm[i:i+batch_size]

            xb = torch.from_numpy(X_train[idx]).float().to(DEVICE)
            yb = torch.from_numpy(y_train[idx]).float().to(DEVICE)

            pred = model(xb)

            loss = criterion(pred, yb)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        val_mae = eval_model_mae(model, X_val, y_val)

        print(f"Epoch {epoch+1} | Val MAE: {val_mae:.4f}")

        if val_mae < best_val:
            best_val = val_mae
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        # else:
        #     patience_counter += 1

        # if patience_counter >= patience:
        #     break

    model.load_state_dict(best_state)
    return model.to(DEVICE), best_val

In [31]:
import numpy as np
import torch

def tft_non_da_val_model_metrics(
    model: torch.nn.Module,
    X: np.ndarray,
    y: np.ndarray,
    batch_size: int = 256
):
    """
    Evaluate model on test data and return MAE, MSE, RMSE
    """

    if X.shape[0] == 0:
        return {"mae": np.nan, "mse": np.nan, "rmse": np.nan}

    model.eval()
    preds = []

    with torch.no_grad():
        for i in range(0, X.shape[0], batch_size):
            xb = torch.from_numpy(X[i : i + batch_size]).float()
            xb = xb.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))

            out = model(xb)  # ✅ single output now
            preds.append(out.cpu().numpy())

    pred = np.concatenate(preds, axis=0).astype(np.float32)

    # 🔥 Metrics
    mae = np.mean(np.abs(pred - y))
    mse = np.mean((pred - y) ** 2)
    rmse = np.sqrt(mse)

    return {
        "mae": float(mae),
        "mse": float(mse),
        "rmse": float(rmse)
    }

## TFT (DA)

### Domain Classifier

In [46]:
class TFTDomainClassifier(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.net(x)

### Gated Residual Network

In [44]:
class TFTGRN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim=None, dropout=0.1):
        super().__init__()
        output_dim = output_dim or input_dim

        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.elu = nn.ELU()
        self.dropout = nn.Dropout(dropout)

        self.gate = nn.Linear(output_dim, output_dim)
        self.sigmoid = nn.Sigmoid()

        self.skip = nn.Linear(input_dim, output_dim) if input_dim != output_dim else None
        self.layer_norm = nn.LayerNorm(output_dim)

    def forward(self, x):
        residual = x

        x = self.elu(self.fc1(x))
        x = self.dropout(self.fc2(x))

        gate = self.sigmoid(self.gate(x))
        x = gate * x

        if self.skip is not None:
            residual = self.skip(residual)

        return self.layer_norm(x + residual)

In [ ]:
def train_tft2_non_da(
    model: nn.Module,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_tgt_val: np.ndarray,
    y_tgt_val: np.ndarray,
    epochs: int = 100,
    batch_size: int = 256,
    lr: float = 1e-3,
    patience: int = 5,
):
    """Train with MSE; early stop on target validation MAE."""
    from torch.utils.data import TensorDataset, DataLoader

    if X_train.shape[0] == 0:
        return model.to(DEVICE), float("nan")

    model = model.to(DEVICE).float()
    train_ds = TensorDataset(torch.from_numpy(X_train).to(DEVICE), torch.from_numpy(y_train).to(DEVICE))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    best_mae, best_state, wait = float("inf"), None, 0

    for ep in range(epochs):
        model.train()
        tr_mse = 0.0
        for xb, yb in train_loader:
            opt.zero_grad() # resetting the gradient from the previous batch. But why?? -> so on the next training iteration the gradient is get set to 0
            # print(model(xb))
            pred,_  = model(xb)
            loss = quantile_loss(pred, yb, model.quantiles)
            # loss = nn.functional.mse_loss(pred, yb)
            loss.backward() # back propagation
            opt.step() # update weights
            tr_mse += loss.item() * xb.size(0) # *** track training loss (accumulates loss accross batches)
        tr_mse /= max(1, X_train.shape[0]) # *** mean training loss per sample

        val_mae = eval_model_tft2_mae(model, X_tgt_val, y_tgt_val, batch_size=256)
        print(f"  Epoch {ep+1}/{epochs}  Train MSE: {tr_mse:.6f}  Target Val MAE: {val_mae:.4f}")

        if val_mae < best_mae:
            best_mae = val_mae
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        #     wait = 0
        # else:
        #     wait += 1
        #     if wait >= patience:
        #         print(f"  Early stopping at epoch {ep+1}")
        #         break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model.to(DEVICE), best_mae


### Variable Selection Network

In [ ]:
import torch.nn.functional as F

class TFTVariableSelectionNetwork(nn.Module):
    def __init__(self, input_dim, num_vars, hidden_dim):
        super().__init__()
        self.num_vars = num_vars

        self.var_grns = nn.ModuleList([
            TFTGRN(input_dim, hidden_dim, hidden_dim)
            for _ in range(num_vars)
        ])

        self.weight_grn = TFTGRN(input_dim * num_vars, hidden_dim, num_vars)

    def forward(self, x):
        # x: [B, T, N, D]
        # B, T, N, D = x.shape
        B, T, N = x.shape

        var_outputs = []
        for i in range(N):
            var_outputs.append(self.var_grns[i](x[:, :, i, :]))

        var_outputs = torch.stack(var_outputs, dim=2)  # [B, T, N, H]

        flat = x.reshape(B, T, -1)
        weights = F.softmax(self.weight_grn(flat), dim=-1).unsqueeze(-1)

        out = (weights * var_outputs).sum(dim=2)

        return out, weights

### Gradient Reversal Layer

In [118]:
class TFTGRL(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_ * grad_output, None

class TFT_DA_Model(nn.Module):
    def __init__(self, tft, hidden_dim):
        super().__init__()
        self.tft = tft

        self.task_head = nn.Linear(hidden_dim, 1)
        self.domain_classifier = TFTDomainClassifier(hidden_dim)

    def forward(self, x, lambda_grl=1.0):
        features, _, _ = self.tft(x)

        # Use last timestep (or pooled)
        feat = features[:, -1, :]

        # Forecast
        y_pred = self.task_head(feat)

        # Domain prediction (with GRL)
        feat_grl = TFTGRL.apply(feat, lambda_grl)
        domain_pred = self.domain_classifier(feat_grl)

        return y_pred, domain_pred

### TFT Backbone/ Feature Extractor

In [49]:
class TFT_Backbone(nn.Module):
    def __init__(
        self,
        input_dim,
        num_vars,
        hidden_dim=64,
        lstm_layers=1,
        num_heads=4,
        dropout=0.1
    ):
        super().__init__()

        self.vsn = TFTVariableSelectionNetwork(input_dim, num_vars, hidden_dim)

        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=lstm_layers,
            batch_first=True
        )

        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            batch_first=True
        )

        self.post_grn = TFTGRN(hidden_dim, hidden_dim)

    def forward(self, x):
        # x: [B, T, N, D]

        x, var_weights = self.vsn(x)

        lstm_out, _ = self.lstm(x)

        attn_out, attn_weights = self.attn(lstm_out, lstm_out, lstm_out)

        features = self.post_grn(attn_out)

        return features, var_weights, attn_weights

### Full Model

In [50]:
class TFT_DA_Model(nn.Module):
    def __init__(
        self,
        input_dim,
        num_vars,
        hidden_dim=64,
        horizon=7
    ):
        super().__init__()

        self.horizon = horizon

        self.backbone = TFT_Backbone(
            input_dim=input_dim,
            num_vars=num_vars,
            hidden_dim=hidden_dim
        )

        # Forecast head (multi-horizon)
        self.forecast_head = nn.Linear(hidden_dim, horizon)

        # Domain classifier
        self.domain_classifier = TFTDomainClassifier(hidden_dim)

    def forward(self, x, lambda_grl=1.0):
        features, var_w, attn_w = self.backbone(x)

        # Pooling (important!)
        feat = features.mean(dim=1)  # [B, H]

        # Forecast
        forecast = self.forecast_head(feat)  # [B, horizon]

        # Domain prediction with GRL
        feat_grl = TFTGRL.apply(feat, lambda_grl)
        domain_pred = self.domain_classifier(feat_grl)

        return forecast, domain_pred, var_w, attn_w

### Eval Tuning TFT DA

In [51]:
def eval_tft_da(model, X, y, batch_size=256):
    model.eval()
    preds = []

    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.from_numpy(X[i:i+batch_size]).to(DEVICE)
            yhat, _, _, _ = model(xb, lambda_grl=0.0)
            preds.append(yhat.cpu())

    preds = torch.cat(preds, dim=0)
    return torch.mean(torch.abs(preds - torch.from_numpy(y))).item()

### Eval Model TFT DA

In [62]:
def tft_eval_model_metrics(model: nn.Module, X: pd.DataFrame, y: pd.DataFrame, batch_size=256):
    model.eval()
    preds = []

    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.from_numpy(X[i:i+batch_size]).to(DEVICE)
            yhat, _, _, _ = model(xb, lambda_grl=0.0)
            preds.append(yhat.cpu())

    preds = torch.cat(preds, dim=0)
    y = torch.from_numpy(y)

    mae = torch.mean(torch.abs(preds - y)).item()
    mse = torch.mean((preds - y) ** 2).item()
    rmse = mse ** 0.5

    return {"mae": mae, "mse": mse, "rmse": rmse}

### Train TFT

In [54]:
def train_tft_da(
    model: nn.Module,
    X_src: np.ndarray,
    y_src: np.ndarray,
    X_tgt: np.ndarray,
    y_tgt: np.ndarray,
    X_tgt_val: np.ndarray,
    y_tgt_val: np.ndarray,
    epochs: int = 100,
    batch_size: int = 256,
    lr: float = 1e-3,
    patience: int = 5,
    use_target_task_loss: bool = False,
):
    from torch.utils.data import TensorDataset, DataLoader

    if X_src.shape[0] == 0 or X_tgt.shape[0] == 0:
        return None, float("nan")

    src_bs = max(1, batch_size // 2)
    tgt_bs = max(1, batch_size // 2)

    src_ds = TensorDataset(torch.from_numpy(X_src).to(DEVICE), torch.from_numpy(y_src).to(DEVICE))
    tgt_ds = TensorDataset(torch.from_numpy(X_tgt).to(DEVICE), torch.from_numpy(y_tgt).to(DEVICE))

    src_loader = DataLoader(src_ds, batch_size=src_bs, shuffle=True, drop_last=True)
    tgt_loader = DataLoader(tgt_ds, batch_size=tgt_bs, shuffle=True, drop_last=True)

    model = model.to(DEVICE).float()
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    total_steps = epochs * min(len(src_loader), len(tgt_loader))
    global_step = 0

    best_mae, best_state, wait = float("inf"), None, 0

    for ep in range(epochs):
        model.train()

        tr_task, tr_dom = 0.0, 0.0
        n_steps = min(len(src_loader), len(tgt_loader))

        src_iter = iter(src_loader)
        tgt_iter = iter(tgt_loader)

        for _ in range(n_steps):
            xs, ys = next(src_iter)
            xt, yt = next(tgt_iter)

            xb = torch.cat([xs, xt], dim=0)

            dom_y = torch.cat([
                torch.zeros(xs.size(0)),
                torch.ones(xt.size(0))
            ]).long().to(DEVICE)

            p = global_step / max(1, total_steps)
            lambd = dann_lambda_schedule(p)
            global_step += 1

            opt.zero_grad()

            # 🔥 TFT forward
            yhat, dom_pred, _, _ = model(xb, lambda_grl=lambd)

            # Task loss
            task_loss = F.mse_loss(yhat[:xs.size(0)], ys)

            if use_target_task_loss:
                task_loss += F.mse_loss(yhat[xs.size(0):], yt)

            # Domain loss
            dom_loss = F.cross_entropy(dom_pred, dom_y)

            loss = task_loss + lambd * dom_loss
            loss.backward()
            opt.step()

            tr_task += task_loss.item()
            tr_dom += dom_loss.item()

        # ✅ Correct evaluation
        val_mae = eval_tft_da(model, X_tgt_val, y_tgt_val)

        tr_task /= max(1, n_steps)
        tr_dom /= max(1, n_steps)

        print(f"Epoch {ep+1}/{epochs} | Task: {tr_task:.6f} | Domain: {tr_dom:.6f} | Val MAE: {val_mae:.4f}")

        if val_mae < best_mae:
            best_mae = val_mae
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        #     wait = 0
        # else:
        #     wait += 1
        #     if wait >= patience:
        #         print(f"Early stopping at epoch {ep+1}")
        #         break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, best_mae

### Run Exp TFT

In [61]:
def run_tft(
    experiment_name: str,
    train_java_df: pd.DataFrame,
    train_papua_df: pd.DataFrame,
    val_papua_df: pd.DataFrame,
    test_papua_df: pd.DataFrame,
    target_station_ids,
    epochs: int = 25,
    batch_size: int = 64,
    lr: float = 1e-3,
    patience: int = 5,
):
    print("\n" + "=" * 60)
    print(f"{experiment_name}")
    print("=" * 60)
    print(f"Selected Papua target stations (train only): {', '.join([str(x) for x in target_station_ids])}")

    train_papua_sel = filter_df_by_station_ids(train_papua_df, target_station_ids)

    X_src, y_src = build_windows(train_java_df)
    X_tgt, y_tgt = build_windows(train_papua_sel)
    X_tgt_val, y_tgt_val = build_windows(val_papua_df)  # FULL Papua val
    X_tgt_test, y_tgt_test = build_windows(test_papua_df)  # FULL Papua test

    print(f"Java train windows:         {X_src.shape[0]}")
    print(f"Papua train windows (sel):  {X_tgt.shape[0]}")
    print(f"Papua val windows (FULL):   {X_tgt_val.shape[0]}")
    print(f"Papua test windows (FULL):  {X_tgt_test.shape[0]}")

    
    # -----------------------------
    # A) TFT - Non-DA
    # -----------------------------
    TFT_metrics = {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")}
    
    if X_src.shape[0] > 0:
        print("\n" + "-" * 60)
        print("A) TFT Non Domain Adaptation")
        tft_model = ClimateTFT(horizon=7, d_model=32, nhead=4, num_layers=4)
        tft_model, best_val_mae = train_tft(
            tft_model,
            X_tgt,
            y_tgt,
            X_tgt_val,
            y_tgt_val,
            epochs=100,
            batch_size=256,
            lr=1e-3,
            patience=5
        )

    TFT_metrics = tft_non_da_val_model_metrics(tft_model, X_tgt_test, y_tgt_test, batch_size=256)

    print(
        f"Papua Test  MAE: {TFT_metrics['mae']:.4f}  "
        f"RMSE: {TFT_metrics['rmse']:.4f}  MSE: {TFT_metrics['mse']:.6f}"
    )

    # -----------------------------
    # B) TFT - DA
    # -----------------------------
    TFT_dann_metrics = {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")}
    if X_src.shape[0] > 0:
        print("\n" + "-" * 60)
        print("B) TFT Domain Adaptation (Java vs selected Papua; balanced batches)")
        print("-" * 60)
        X_src = X_src.reshape(X_src.shape[0], X_src.shape[1], 9, 1)
        X_tgt = X_tgt.reshape(X_tgt.shape[0], X_tgt.shape[1], 9, 1)
        X_tgt_val = X_tgt_val.reshape(X_tgt_val.shape[0], X_tgt_val.shape[1], 9, 1)
        X_tgt_test = X_tgt_test.reshape(X_tgt_test.shape[0], X_tgt_test.shape[1], 9, 1)
        TFT_dann_model = TFT_DA_Model(
            input_dim=X_src.shape[3],
            num_vars=X_src.shape[2],
            hidden_dim=64,
            horizon=HORIZON
        )
    TFT_dann_model, best_val_mae = train_tft_da(
        TFT_dann_model,
        X_src,
        y_src,
        X_tgt,
        y_tgt,
        X_tgt_val,
        y_tgt_val,
        epochs=epochs,
        batch_size=batch_size,
        lr=lr,
        patience=patience,
    ) 

    TFT_dann_metrics = tft_eval_model_metrics(TFT_dann_model, X_tgt_test, y_tgt_test, batch_size=256)
    # print(f"Best Papua Val MAE (early stop): {best_val_mae:.4f}")
    print(
        f"Papua Test  MAE: {TFT_dann_metrics['mae']:.4f}  "
        f"RMSE: {TFT_dann_metrics['rmse']:.4f}  MSE: {TFT_dann_metrics['mse']:.6f}"
    )
    out = {
        "TFT Non Domain adaptatoin": TFT_metrics,
        "TFT_dann": TFT_dann_metrics,
    }

    print("\n" + "-" * 60)
    print("Result (Papua test):")
    for k in ["TFT Non Domain adaptatoin","TFT_dann"]:
        m = out[k]
        print(f"  {k:>18s}  MAE={m['mae']:.4f}  MSE={m['mse']:.6f}  RMSE={m['rmse']:.4f}")
    return out



In [10]:
train, val, test = load_and_split("../data/merged_dataset.csv")
train_java = train[region_mask(train, "java")]
train_papua = train[region_mask(train, "papua")]
val_papua = val[region_mask(val, "papua")]
test_papua = test[region_mask(test, "papua")]

X_src, y_src = build_windows(train_java)
X_tgt, y_tgt = build_windows(train_papua)
X_tgt_val, y_tgt_val = build_windows(val_papua)  # FULL Papua val
X_tgt_test, y_tgt_test = build_windows(test_papua)  

single_id, three_ids, stats_sorted = select_target_stations_papua_by_elevation(train_papua)


In [64]:
res_single = run_tft(
        "Single-Station Experiment (Target=Papua; Train target=1 station; Val/Test=FULL Papua)",
        train_java_df=train_java,
        train_papua_df=train_papua,
        val_papua_df=val_papua,
        test_papua_df=test_papua,
        target_station_ids=[single_id],
        epochs=100,
        batch_size=256,
        lr=1e-3,
        patience=5,
)


Single-Station Experiment (Target=Papua; Train target=1 station; Val/Test=FULL Papua)
Selected Papua target stations (train only): P11
Java train windows:         53487
Papua train windows (sel):  2547
Papua val windows (FULL):   8070
Papua test windows (FULL):  8970

------------------------------------------------------------
A) TFT Non Domain Adaptation
Epoch 1 | Val MAE: 26.7330
Epoch 2 | Val MAE: 26.0345
Epoch 3 | Val MAE: 25.3732
Epoch 4 | Val MAE: 24.6708
Epoch 5 | Val MAE: 23.9065
Epoch 6 | Val MAE: 23.0951
Epoch 7 | Val MAE: 22.2326
Epoch 8 | Val MAE: 21.3241
Epoch 9 | Val MAE: 20.3756
Epoch 10 | Val MAE: 19.3942
Epoch 11 | Val MAE: 18.3858
Epoch 12 | Val MAE: 17.3606
Epoch 13 | Val MAE: 16.3222
Epoch 14 | Val MAE: 15.2785
Epoch 15 | Val MAE: 14.2332
Epoch 16 | Val MAE: 13.1937
Epoch 17 | Val MAE: 12.1662
Epoch 18 | Val MAE: 11.1588
Epoch 19 | Val MAE: 10.1788
Epoch 20 | Val MAE: 9.2306
Epoch 21 | Val MAE: 8.3220
Epoch 22 | Val MAE: 7.4634
Epoch 23 | Val MAE: 6.6577
Epoch 24 

In [65]:
res_three = run_tft(
        "Single-Station Experiment (Target=Papua; Train target=1 station; Val/Test=FULL Papua)",
        train_java_df=train_java,
        train_papua_df=train_papua,
        val_papua_df=val_papua,
        test_papua_df=test_papua,
        target_station_ids=three_ids,
        epochs=100,
        batch_size=256,
        lr=1e-3,
        patience=5,
)


Single-Station Experiment (Target=Papua; Train target=1 station; Val/Test=FULL Papua)
Selected Papua target stations (train only): P08, P11, P14
Java train windows:         53487
Papua train windows (sel):  7641
Papua val windows (FULL):   8070
Papua test windows (FULL):  8970

------------------------------------------------------------
A) TFT Non Domain Adaptation
Epoch 1 | Val MAE: 25.3320
Epoch 2 | Val MAE: 23.0572
Epoch 3 | Val MAE: 20.3426
Epoch 4 | Val MAE: 17.3398
Epoch 5 | Val MAE: 14.2089
Epoch 6 | Val MAE: 11.1199
Epoch 7 | Val MAE: 8.2744
Epoch 8 | Val MAE: 5.8826
Epoch 9 | Val MAE: 4.1425
Epoch 10 | Val MAE: 3.1194
Epoch 11 | Val MAE: 2.6470
Epoch 12 | Val MAE: 2.2633
Epoch 13 | Val MAE: 1.8231
Epoch 14 | Val MAE: 1.6382
Epoch 15 | Val MAE: 1.8583
Epoch 16 | Val MAE: 1.6910
Epoch 17 | Val MAE: 1.7094
Epoch 18 | Val MAE: 1.7251
Epoch 19 | Val MAE: 1.8689
Epoch 20 | Val MAE: 1.8490
Epoch 21 | Val MAE: 1.8198
Epoch 22 | Val MAE: 1.8068
Epoch 23 | Val MAE: 1.7479
Epoch 24 | V

In [74]:
def _fmt(x):
    return f"{x:7.3f}" if (x is not None and not np.isnan(x)) else " nan"


def print_summary_table(res_single: dict, res_three: dict):
    print("\nSpatial Scarcity Summary Table (Papua test MAE)")
    print("=" * 60)
    methods = [k for k,v in res_single.items()]
    experiments = [
        ("Single-Station", res_single),
        ("Three-Station", res_three)
    ]
    
    print("Experiment        | " + " | ".join([f"{m} MAE".ljust(15) for m in methods]))
    print("-" * 60)
    for exp_name, res in experiments:
        row = " | ".join([f"{_fmt(res[m]['mae'])}".ljust(15) for m in methods])
        print(f"{exp_name:<17}" + " | " + f"{row}")
    
    print("-" * 60)
    print("\n" + "-" * 60)

    print("Experiment        | " + " | ".join([f"{m} MSE".ljust(15) for m in methods]))
    print("-" * 60)
    for exp_name, res in experiments:
        row = " | ".join([f"{_fmt(res[m]['mse'])}".ljust(15) for m in methods])
        print(f"{exp_name:<17}" + " | " + f"{row}")


    print("-" * 60)
    print("\n" + "-" * 60)

    print("Experiment        | " + " | ".join([f"{m} RMSE".ljust(15) for m in methods]))
    print("-" * 60)
    for exp_name, res in experiments:
        row = " | ".join([f"{_fmt(res[m]['rmse'])}".ljust(15) for m in methods])
        print(f"{exp_name:<17}" + " | " + f"{row}")

    print("-" * 60)
# print_summary_table(res_single, res_three)

## TFT2 Non DA

In [11]:
class TemporalAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.attn = nn.Linear(d_model, 1)

    def forward(self, x):
        # x: (B, T, d_model)
        scores = self.attn(x)                     # (B, T, 1)
        weights = torch.softmax(scores, dim=1)    # attention over time
        context = (weights * x).sum(dim=1)        # weighted sum
        return context, weights

In [12]:
def quantile_loss(preds, target, quantiles):
    """
    preds: (B, H, Q)
    target: (B, H)
    """
    losses = []
    for i, q in enumerate(quantiles):
        errors = target - preds[:, :, i]
        loss = torch.max((q - 1) * errors, q * errors)
        losses.append(loss.unsqueeze(-1))
    return torch.mean(torch.cat(losses, dim=-1))

In [13]:
class HorizonDecoder(nn.Module):
    def __init__(self, d_model, horizon, num_quantiles):
        super().__init__()
        self.horizon = horizon
        self.query = nn.Parameter(torch.randn(horizon, d_model))

        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=4,
            batch_first=True
        )

        self.proj = nn.Linear(d_model, num_quantiles)

    def forward(self, context_seq):
        """
        context_seq: (B, T, d_model)
        """
        B = context_seq.size(0)

        # expand learned queries
        q = self.query.unsqueeze(0).expand(B, -1, -1)  # (B, H, d_model)

        # cross-attention: horizon queries attend to time
        out, _ = self.attn(q, context_seq, context_seq)

        return self.proj(out)  # (B, H, Q)

In [67]:
class VariableSelectionNetwork(nn.Module):
    def __init__(self, num_vars, d_model):
        super().__init__()
        self.weight_net = nn.Sequential(
            nn.Linear(num_vars, num_vars),
            nn.Softmax(dim=-1)
        )
        self.proj = nn.Linear(num_vars, d_model)

    def forward(self, x):
        # x: (B, T, num_vars)
        weights = self.weight_net(x.mean(dim=1))  # (B, num_vars)
        x_weighted = x * weights.unsqueeze(1)
        return self.proj(x_weighted)

# class VariableSelectionNetwork(nn.Module):
#     def __init__(self, num_vars, d_model, hidden_dim=64, dropout=0.1):
#         super().__init__()
#         self.num_vars = num_vars
#         self.d_model = d_model

#         # 🔹 Variable-wise GRNs (one per variable)
#         self.var_grns = nn.ModuleList([
#             GRN(1, hidden_dim, d_model, dropout)
#             for _ in range(num_vars)
#         ])

#         # 🔹 Weighting network (importance of variables)
#         self.weight_grn = GRN(num_vars, hidden_dim, num_vars, dropout)

#         self.softmax = nn.Softmax(dim=-1)

#     def forward(self, x):
#         """
#         x: (B, T, num_vars)
#         """
#         B, T, N = x.shape

#         if N != self.num_vars:
#             raise ValueError(f"Expected {self.num_vars}, got {N}")

#         # 🔹 Step 1: process each variable independently
#         var_outputs = []
#         for i in range(N):
#             xi = x[:, :, i].unsqueeze(-1)      # (B, T, 1)
#             vi = self.var_grns[i](xi)          # (B, T, d_model)
#             var_outputs.append(vi)

#         # stack: (B, T, N, d_model)
#         var_stack = torch.stack(var_outputs, dim=2)

#         # 🔹 Step 2: compute variable importance weights
#         # use temporal context
#         context = x.mean(dim=1)               # (B, N)

#         weights = self.weight_grn(context)    # (B, N)
#         weights = self.softmax(weights)       # (B, N)

#         # 🔹 Step 3: apply weights
#         weights = weights.unsqueeze(1).unsqueeze(-1)  # (B, 1, N, 1)

#         weighted = var_stack * weights        # (B, T, N, d_model)

#         # 🔹 Step 4: sum over variables
#         out = weighted.sum(dim=2)             # (B, T, d_model)

#         return out

In [68]:
class GRN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim=None, dropout=0.1):
        super().__init__()
        output_dim = output_dim or input_dim

        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.elu = nn.ELU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)

        self.dropout = nn.Dropout(dropout)
        self.gate = nn.Linear(output_dim, output_dim)
        self.norm = nn.LayerNorm(output_dim)

        if input_dim != output_dim:
            self.skip = nn.Linear(input_dim, output_dim)
        else:
            self.skip = None

    def forward(self, x):
        residual = x if self.skip is None else self.skip(x)

        out = self.fc1(x)
        out = self.elu(out)
        out = self.fc2(out)
        out = self.dropout(out)

        gate = torch.sigmoid(self.gate(out))
        out = gate * out + (1 - gate) * residual

        return self.norm(out)

# import torch
# import torch.nn as nn
# import torch.nn.functional as F


# class GRN(nn.Module):
#     def __init__(self, input_dim, hidden_dim, output_dim=None, dropout=0.1):
#         super().__init__()
#         self.output_dim = output_dim if output_dim is not None else input_dim

#         self.fc1 = nn.Linear(input_dim, hidden_dim)
#         self.elu = nn.ELU()
#         self.fc2 = nn.Linear(hidden_dim, self.output_dim)

#         self.dropout = nn.Dropout(dropout)

#         # gating
#         self.gate = nn.Linear(self.output_dim, self.output_dim)
#         self.sigmoid = nn.Sigmoid()

#         # skip connection projection (if needed)
#         if input_dim != self.output_dim:
#             self.skip = nn.Linear(input_dim, self.output_dim)
#         else:
#             self.skip = None

#         self.layer_norm = nn.LayerNorm(self.output_dim)

#     def forward(self, x):
#         residual = x

#         x = self.fc1(x)
#         x = self.elu(x)
#         x = self.fc2(x)
#         x = self.dropout(x)

#         # gating
#         gate = self.sigmoid(self.gate(x))
#         x = x * gate

#         # residual connection
#         if self.skip is not None:
#             residual = self.skip(residual)

#         return self.layer_norm(x + residual)

In [69]:
class ClimateTFTv2(nn.Module):
    def __init__(
        self,
        horizon=7,
        d_model=32,
        nhead=4,
        num_layers=2,
        dropout=0.1,
        quantiles=[0.1, 0.5, 0.9],
    ):
        super().__init__()

        self.quantiles = quantiles
        self.num_q = len(quantiles)

        self.idx_temp = [FEATURE_COLS.index(c) for c in LOCAL_TEMPORAL_COLS]
        self.idx_geo = [FEATURE_COLS.index(c) for c in GEO_COLS]
        self.idx_clim = [FEATURE_COLS.index(c) for c in CLIMATE_COLS]

        # --- Variable Selection ---
        self.vsn = VariableSelectionNetwork(len(self.idx_temp), d_model)

        # --- Transformer ---
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # --- Attention over time ---
        self.temporal_attn = TemporalAttention(d_model)

        # --- Static encoders ---
        self.geo_grn = GRN(3, 32, d_model)
        self.clim_grn = GRN(2, 32, d_model)

        # --- Fusion ---
        self.fusion_grn = GRN(d_model * 3, 64, d_model)

        # --- Decoder ---
        self.decoder = HorizonDecoder(d_model, horizon, self.num_q)

    def forward(self, x):
        xt = x[:, :, self.idx_temp]
        xg = x[:, 0, self.idx_geo]
        xc = x[:, 0, self.idx_clim]

        # --- Variable selection ---
        xt = self.vsn(xt)

        # --- Transformer ---
        h_seq = self.transformer(xt)

        # --- Temporal attention ---
        h_context, attn_weights = self.temporal_attn(h_seq)

        # --- Static features ---
        g = self.geo_grn(xg)
        c = self.clim_grn(xc)

        # --- Fusion ---
        fused = torch.cat([h_context, g, c], dim=1)
        fused = self.fusion_grn(fused)

        # Expand fused to sequence for decoder
        fused_seq = fused.unsqueeze(1).repeat(1, h_seq.size(1), 1)

        # Combine with temporal sequence
        full_seq = h_seq + fused_seq

        # --- Decoder ---
        out = self.decoder(full_seq)  # (B, H, Q)

        return out, attn_weights

### TFT2 Eval Tuning Model Mae

In [31]:
def tft2_eval_model_mae(model: torch.nn.Module, X: np.ndarray, y: np.ndarray, batch_size: int = 256) -> float:
    if X.shape[0] == 0:
        return float("nan")

    model.eval()
    preds = []

    median_idx = model.quantiles.index(0.5)

    with torch.no_grad():
        for i in range(0, X.shape[0], batch_size):
            xb = torch.from_numpy(X[i : i + batch_size])
            xb = xb.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
            # 🔥 FIX HERE
            # yhat, _ , _= model(xb, lambda_grl=0.0)
            yhat, _  = model(xb)
            yhat = yhat[:,:, median_idx]

            preds.append(yhat.cpu().numpy().astype(np.float32))

    pred = np.concatenate(preds, axis=0)
    return float(np.mean(np.abs(pred - y)))

In [32]:
def tft2_eval_model_metrics(model, X, y, batch_size=256):
    model.eval()
    preds = []

    median_idx = model.quantiles.index(0.5)

    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.from_numpy(X[i:i+batch_size]).to(DEVICE)
            yhat, _ = model(xb)

            yhat = yhat[:, :, median_idx]  # (B, H)
            preds.append(yhat.cpu())

    preds = torch.cat(preds, dim=0)
    y = torch.from_numpy(y)

    mae = torch.mean(torch.abs(preds - y)).item()
    mse = torch.mean((preds - y) ** 2).item()
    rmse = mse ** 0.5

    return {"mae": mae, "mse": mse, "rmse": rmse}

### Train TFT2 Non-DA

In [19]:
def train_tft2_non_da(
    model: nn.Module,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_tgt_val: np.ndarray,
    y_tgt_val: np.ndarray,
    epochs: int = 100,
    batch_size: int = 256,
    lr: float = 1e-3,
    patience: int = 5,
):
    """Train with MSE; early stop on target validation MAE."""
    from torch.utils.data import TensorDataset, DataLoader

    if X_train.shape[0] == 0:
        return model.to(DEVICE), float("nan")

    model = model.to(DEVICE).float()
    train_ds = TensorDataset(torch.from_numpy(X_train).to(DEVICE), torch.from_numpy(y_train).to(DEVICE))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    best_mae, best_state, wait = float("inf"), None, 0

    for ep in range(epochs):
        model.train()
        tr_mse = 0.0
        for xb, yb in train_loader:
            opt.zero_grad() # resetting the gradient from the previous batch. But why?? -> so on the next training iteration the gradient is get set to 0
            # print(model(xb))
            pred,_  = model(xb)
            loss = quantile_loss(pred, yb, model.quantiles)
            # loss = nn.functional.mse_loss(pred, yb)
            loss.backward() # back propagation
            opt.step() # update weights
            tr_mse += loss.item() * xb.size(0) # *** track training loss (accumulates loss accross batches)
        tr_mse /= max(1, X_train.shape[0]) # *** mean training loss per sample

        val_mae = tft2_eval_model_mae(model, X_tgt_val, y_tgt_val, batch_size=256)
        print(f"  Epoch {ep+1}/{epochs}  Train MSE: {tr_mse:.6f}  Target Val MAE: {val_mae:.4f}")

        if val_mae < best_mae:
            best_mae = val_mae
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        #     wait = 0
        # else:
        #     wait += 1
        #     if wait >= patience:
        #         print(f"  Early stopping at epoch {ep+1}")
        #         break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model.to(DEVICE), best_mae


## TFT2 Domain Adaptatation

### Gradient Reversal Layer

In [20]:
class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_ * grad_output, None


def grad_reverse(x, lambda_=1.0):
    return GradReverse.apply(x, lambda_)

### Domain Classifier

In [21]:
class DomainClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 2)  # source vs target
        )

    def forward(self, x):
        return self.net(x)

### Full Model

In [57]:
class ClimateTFTv2_DA(ClimateTFTv2):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.domain_clf = DomainClassifier(32)

    def forward(self, x, lambda_grl=0.0):
        xt = x[:, :, self.idx_temp]
        xg = x[:, 0, self.idx_geo]
        xc = x[:, 0, self.idx_clim]

        xt = self.vsn(xt)
        h_seq = self.transformer(xt)

        h_context, attn_weights = self.temporal_attn(h_seq)

        g = self.geo_grn(xg)
        c = self.clim_grn(xc)

        fused = torch.cat([h_context, g, c], dim=1)
        fused = self.fusion_grn(fused)

        # --- Forecast ---
        fused_seq = fused.unsqueeze(1).repeat(1, h_seq.size(1), 1)
        full_seq = h_seq + fused_seq
        forecast = self.decoder(full_seq)

        # --- Domain prediction (GRL applied) ---
        rev_feat = grad_reverse(fused, lambda_grl)
        domain_logits = self.domain_clf(rev_feat)

        # return forecast, domain_logits, attn_weights
        return forecast, domain_logits

### Eval Tuning TFT2 DA (Already declared before)

In [23]:
def eval_tft2_DA_model_mae(model: torch.nn.Module, X: np.ndarray, y: np.ndarray, batch_size: int = 256) -> float:
    if X.shape[0] == 0:
        return float("nan")

    model.eval()
    preds = []

    median_idx = model.quantiles.index(0.5)

    with torch.no_grad():
        for i in range(0, X.shape[0], batch_size):
            xb = torch.from_numpy(X[i : i + batch_size])
            xb = xb.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
            # 🔥 FIX HERE
            # yhat, _ , _= model(xb, lambda_grl=0.0)
            yhat, _, _ = model(xb)
            yhat = yhat[:,:, median_idx]

            preds.append(yhat.cpu().numpy().astype(np.float32))

    pred = np.concatenate(preds, axis=0)
    return float(np.mean(np.abs(pred - y)))

### Eval Model TFT2 DA (already declared before)

In [24]:
def tft2_eval_model_metrics(model, X, y, batch_size=256):
    model.eval()
    preds = []

    median_idx = model.quantiles.index(0.5)

    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.from_numpy(X[i:i+batch_size]).to(DEVICE)
            yhat, _ , _= model(xb)

            yhat = yhat[:, :, median_idx]  # (B, H)
            preds.append(yhat.cpu())

    preds = torch.cat(preds, dim=0)
    y = torch.from_numpy(y)

    mae = torch.mean(torch.abs(preds - y)).item()
    mse = torch.mean((preds - y) ** 2).item()
    rmse = mse ** 0.5

    return {"mae": mae, "mse": mse, "rmse": rmse}

### Train tft2 DA

In [64]:
def train_tft2_da(
    model: nn.Module,
    X_src: pd.DataFrame, 
    y_src: pd.DataFrame,
    X_tgt: pd.DataFrame,
    X_tgt_val: pd.DataFrame,
    y_tgt_val: pd.DataFrame,  
    epochs: int =100,
    batch_size: int =256,
    lambda_max: float=1.0,
    lr:float = 1e-3,
    patience: int=5
):
    from torch.utils.data import TensorDataset, DataLoader
    import torch.nn.functional as F

    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    src_ds = TensorDataset(torch.from_numpy(X_src), torch.from_numpy(y_src))
    tgt_ds = TensorDataset(torch.from_numpy(X_tgt))

    src_loader = DataLoader(src_ds, batch_size=batch_size,  num_workers=0)
    tgt_loader = DataLoader(tgt_ds, batch_size=batch_size,  num_workers=0)

    best_mae = float("inf")
    best_state = None
    wait = 0

    for ep in range(epochs):
        model.train()
        tr_task, tr_dom = 0.0, 0.0
        lambda_grl = lambda_max * (ep / epochs)

        for (xs, ys), (xt,) in zip(src_loader, tgt_loader):
            xs, ys = xs.to(DEVICE), ys.to(DEVICE)
            xt = xt.to(DEVICE)

            # --- Forward ---
            pred_s, dom_s= model(xs, lambda_grl)
            _, dom_t = model(xt, lambda_grl)
            # pred_s, dom_s, _ = model(xs, lambda_grl)
            # _, dom_t, _ = model(xt, lambda_grl)

            # --- Losses ---
            loss_forecast = quantile_loss(pred_s, ys, model.quantiles)

            dom_labels_s = torch.zeros(dom_s.size(0), dtype=torch.long).to(DEVICE)
            dom_labels_t = torch.ones(dom_t.size(0), dtype=torch.long).to(DEVICE)

            loss_domain = (
                F.cross_entropy(dom_s, dom_labels_s) +
                F.cross_entropy(dom_t, dom_labels_t)
            )

            loss = loss_forecast + 0.1 * loss_domain

            # --- Backprop ---
            opt.zero_grad()
            loss.backward()
            opt.step()

            tr_task += loss_forecast.item()
            tr_dom += loss_domain.item()

        # ✅ VALIDATION (Papua)
        val_mae = tft2_eval_model_mae(model, X_tgt_val, y_tgt_val, batch_size=256)

        print(f"Epoch {ep+1} | Task: {tr_task:.4f} | Domain: {tr_dom:.6f} | Val MAE: {val_mae}")

        # --- Early stopping ---
        if val_mae < best_mae:
            best_mae = val_mae
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        #     wait = 0
        # else:
        #     wait += 1
        #     if wait >= patience:
        #         print(f"Early stopping at epoch {ep+1}")
        #         break

    # restore best model
    if best_state is not None:
        model.load_state_dict(best_state)

    return model.to(DEVICE), best_mae

In [63]:
## test train TFT2

def train_dann_tft2(
    model,
    X_src, y_src,
    X_tgt,
    X_tgt_val, y_tgt_val,   # 👈 ADD THIS
    epochs=25,
    batch_size=64,
    lambda_max=1.0,
    patience=5
):
    from torch.utils.data import TensorDataset, DataLoader
    import torch.nn.functional as F

    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    src_ds = TensorDataset(torch.from_numpy(X_src), torch.from_numpy(y_src))
    tgt_ds = TensorDataset(torch.from_numpy(X_tgt))

    src_loader = DataLoader(src_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    tgt_loader = DataLoader(tgt_ds, batch_size=batch_size, shuffle=True, num_workers=0)

    best_mae = float("inf")
    best_state = None
    wait = 0

    for ep in range(epochs):
        model.train()

        lambda_grl = lambda_max * (ep / epochs)

        for (xs, ys), (xt,) in zip(src_loader, tgt_loader):
            xs, ys = xs.to(DEVICE), ys.to(DEVICE)
            xt = xt.to(DEVICE)

            # --- Forward ---
            pred_s, dom_s = model(xs, lambda_grl)
            _, dom_t = model(xt, lambda_grl)
            # pred_s, dom_s, _ = model(xs, lambda_grl)
            # _, dom_t, _ = model(xt, lambda_grl)

            # --- Losses ---
            loss_forecast = quantile_loss(pred_s, ys, model.quantiles)

            dom_labels_s = torch.zeros(dom_s.size(0), dtype=torch.long).to(DEVICE)
            dom_labels_t = torch.ones(dom_t.size(0), dtype=torch.long).to(DEVICE)

            loss_domain = (
                F.cross_entropy(dom_s, dom_labels_s) +
                F.cross_entropy(dom_t, dom_labels_t)
            )

            loss = loss_forecast + 0.1 * loss_domain

            # --- Backprop ---
            opt.zero_grad()
            loss.backward()
            opt.step()

        # ✅ VALIDATION (Papua)
        val_mae = tft2_eval_model_metrics(model, X_tgt_val, y_tgt_val)["mae"]

        print(f"Epoch {ep+1} | Val MAE: {val_mae:.4f}")

        # --- Early stopping ---
        if val_mae < best_mae:
            best_mae = val_mae
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping at epoch {ep+1}")
                break

    # restore best model
    if best_state is not None:
        model.load_state_dict(best_state)

    return model.to(DEVICE), best_mae

### EXP TFT 2 DA and Non-DA

In [ ]:
def run_tft_2(
    experiment_name: str,
    train_java_df: pd.DataFrame,
    train_papua_df: pd.DataFrame,
    val_papua_df: pd.DataFrame,
    test_papua_df: pd.DataFrame,
    target_station_ids,
    epochs: int = 100,
    batch_size: int = 256,
    lr: float = 1e-3,
    patience: int = 5,
):
    print("\n" + "=" * 60)
    print(f"{experiment_name}")
    print("=" * 60)
    print(f"Selected Papua target stations (train only): {', '.join([str(x) for x in target_station_ids])}")

    train_papua_sel = filter_df_by_station_ids(train_papua_df, target_station_ids)

    X_src, y_src = build_windows(train_java_df)
    X_tgt, y_tgt = build_windows(train_papua_sel)
    X_tgt_val, y_tgt_val = build_windows(val_papua_df)  # FULL Papua val
    X_tgt_test, y_tgt_test = build_windows(test_papua_df)  # FULL Papua test

    print(f"Java train windows:         {X_src.shape[0]}")
    print(f"Papua train windows (sel):  {X_tgt.shape[0]}")
    print(f"Papua val windows (FULL):   {X_tgt_val.shape[0]}")
    print(f"Papua test windows (FULL):  {X_tgt_test.shape[0]}")

    
    # -----------------------------
    # A) TFT2 - Non-DA
    # -----------------------------
    TFT2_metrics = {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")}
    
    if X_src.shape[0] > 0:
        print("\n" + "-" * 60)
        print("A) TFT 2 Non Domain Adaptation")
        tft2_model = ClimateTFTv2(horizon=7, d_model=32, nhead=4, num_layers=2, dropout=0.1, quantiles=[0.1, 0.5, 0.9])
        tft2_model, _ = train_tft2_non_da(
            tft2_model,
            X_tgt,
            y_tgt,
            X_tgt_val,
            y_tgt_val,
            epochs=100,
            batch_size=256,
            lr=1e-3,
            patience=5
        )

    TFT2_metrics = tft2_eval_model_metrics(tft2_model, X_tgt_test, y_tgt_test, batch_size=256)

    print(
        f"Papua Test  MAE: {TFT2_metrics['mae']:.4f}  "
        f"RMSE: {TFT2_metrics['rmse']:.4f}  MSE: {TFT2_metrics['mse']:.6f}"
    )

    # -----------------------------
    # B) TFT - DA
    # -----------------------------
    TFT2_dann_metrics = {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")}
    if X_src.shape[0] > 0:
        print("\n" + "-" * 60)
        print("B) TFT Domain Adaptation (Java vs selected Papua; balanced batches)")
        print("-" * 60)
        # X_src = X_src.reshape(X_src.shape[0], X_src.shape[1], 9, 1)
        # X_tgt = X_tgt.reshape(X_tgt.shape[0], X_tgt.shape[1], 9, 1)
        # X_tgt_val = X_tgt_val.reshape(X_tgt_val.shape[0], X_tgt_val.shape[1], 9, 1)
        # X_tgt_test = X_tgt_test.reshape(X_tgt_test.shape[0], X_tgt_test.shape[1], 9, 1)
        TFT2_dann_model = ClimateTFTv2_DA(
            horizon=7, 
            d_model=32,
            nhead=4,
            num_layers=2,
            dropout=0.1,
            quantiles=[0.1, 0.5, 0.9]
        )
    TFT2_dann_model, _ = train_tft2_da(
        TFT2_dann_model,
        X_src,
        y_src,
        X_tgt,
        X_tgt_val,
        y_tgt_val,
        epochs=100,
        batch_size=256,
        lambda_max=1.0,
        lr=1e-3,
        patience=5,
    ) 

    TFT2_dann_metrics = tft2_eval_model_metrics(TFT2_dann_model, X_tgt_test, y_tgt_test, batch_size=256)
    # print(f"Best Papua Val MAE (early stop): {best_val_mae:.4f}")
    print(
        f"Papua Test  MAE: {TFT2_dann_metrics['mae']:.4f}  "
        f"RMSE: {TFT2_dann_metrics['rmse']:.4f}  MSE: {TFT2_dann_metrics['mse']:.6f}"
    )
    out = {
        "TFT2 Non Domain adaptatoin": TFT2_metrics,
        "TFT2_dann": TFT2_dann_metrics,
    }

    print("\n" + "-" * 60)
    print("Result (Papua test):")
    for k in ["TFT2 Non Domain adaptatoin","TFT2_dann"]:
        m = out[k]
        print(f"  {k:>18s}  MAE={m['mae']:.4f}  MSE={m['mse']:.6f}  RMSE={m['rmse']:.4f}")
    return out



In [70]:
res_single_tft2 = run_tft_2(
    "Single-Station Experiment (Target=Papua; Train target=1 station; Val/Test=FULL Papua)",
        train_java_df=train_java,
        train_papua_df=train_papua,
        val_papua_df=val_papua,
        test_papua_df=test_papua,
        target_station_ids=[single_id],
        epochs=10,
        batch_size=256,
        lr=1e-3,
        patience=5,
)


Single-Station Experiment (Target=Papua; Train target=1 station; Val/Test=FULL Papua)
Selected Papua target stations (train only): P11
Java train windows:         53487
Papua train windows (sel):  2547
Papua val windows (FULL):   8070
Papua test windows (FULL):  8970

------------------------------------------------------------
A) TFT 2 Non Domain Adaptation
  Epoch 1/100  Train MSE: 13.514102  Target Val MAE: 26.6610
  Epoch 2/100  Train MSE: 11.952083  Target Val MAE: 24.3632
  Epoch 3/100  Train MSE: 9.493160  Target Val MAE: 20.1533
  Epoch 4/100  Train MSE: 5.223904  Target Val MAE: 12.4279
  Epoch 5/100  Train MSE: 2.121396  Target Val MAE: 2.7580
  Epoch 6/100  Train MSE: 0.947278  Target Val MAE: 2.0218
  Epoch 7/100  Train MSE: 0.617922  Target Val MAE: 2.1178
  Epoch 8/100  Train MSE: 0.447469  Target Val MAE: 1.9823
  Epoch 9/100  Train MSE: 0.324709  Target Val MAE: 1.9377
  Epoch 10/100  Train MSE: 0.287119  Target Val MAE: 2.0082
  Epoch 11/100  Train MSE: 0.278084  Targ

In [71]:
res_three_tft2 = run_tft_2(
    "Single-Station Experiment (Target=Papua; Train target=1 station; Val/Test=FULL Papua)",
        train_java_df=train_java,
        train_papua_df=train_papua,
        val_papua_df=val_papua,
        test_papua_df=test_papua,
        target_station_ids=three_ids,
        epochs=10,
        batch_size=256,
        lr=1e-3,
        patience=5,
)


Single-Station Experiment (Target=Papua; Train target=1 station; Val/Test=FULL Papua)
Selected Papua target stations (train only): P08, P11, P14
Java train windows:         53487
Papua train windows (sel):  7641
Papua val windows (FULL):   8070
Papua test windows (FULL):  8970

------------------------------------------------------------
A) TFT 2 Non Domain Adaptation
  Epoch 1/100  Train MSE: 11.625494  Target Val MAE: 20.3405
  Epoch 2/100  Train MSE: 3.136334  Target Val MAE: 1.5799
  Epoch 3/100  Train MSE: 0.443878  Target Val MAE: 1.5180
  Epoch 4/100  Train MSE: 0.308510  Target Val MAE: 1.7833
  Epoch 5/100  Train MSE: 0.295004  Target Val MAE: 1.7661
  Epoch 6/100  Train MSE: 0.286684  Target Val MAE: 1.7475
  Epoch 7/100  Train MSE: 0.282197  Target Val MAE: 1.7065
  Epoch 8/100  Train MSE: 0.282961  Target Val MAE: 1.7955
  Epoch 9/100  Train MSE: 0.273994  Target Val MAE: 1.7116
  Epoch 10/100  Train MSE: 0.271176  Target Val MAE: 1.7780
  Epoch 11/100  Train MSE: 0.270599

In [75]:
print_summary_table(res_single_tft2, res_three_tft2)


Spatial Scarcity Summary Table (Papua test MAE)
Experiment        | TFT2 Non Domain adaptatoin MAE | TFT2_dann MAE  
------------------------------------------------------------
Single-Station    |   1.920         |   1.426        
Three-Station     |   1.385         |   1.894        
------------------------------------------------------------

------------------------------------------------------------
Experiment        | TFT2 Non Domain adaptatoin MSE | TFT2_dann MSE  
------------------------------------------------------------
Single-Station    |   5.905         |   3.318        
Three-Station     |   3.006         |   5.819        
------------------------------------------------------------

------------------------------------------------------------
Experiment        | TFT2 Non Domain adaptatoin RMSE | TFT2_dann RMSE 
------------------------------------------------------------
Single-Station    |   2.430         |   1.821        
Three-Station     |   1.734         |   2.41

## KMM Vu Tran

### Projection

In [78]:
class Projection(nn.Module):
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, x):
        # x: (B, T, in_dim)
        x = self.linear(x)
        x = self.norm(x)
        x = F.gelu(x)
        return x  # (B, T, hidden_dim)

### Full Transformer Model

In [79]:
class TemperatureTransformer(nn.Module):
    def __init__(
        self,
        input_dim_primary=1,     # temperature
        input_dim_cov=8,         # covariates
        hidden_dim=64,
        n_heads=4,
        num_layers=2,
        forecast_horizon=7
    ):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.forecast_horizon = forecast_horizon

        # 🔹 Projection layers
        self.primary_proj = Projection(input_dim_primary, hidden_dim)
        self.cov_proj = Projection(input_dim_cov, hidden_dim)

        # 🔹 Aggregation projection
        self.agg_proj = Projection(hidden_dim * 2, hidden_dim)

        # 🔹 Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=n_heads,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        # 🔹 Transformer Decoder
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=hidden_dim,
            nhead=n_heads,
            batch_first=True
        )
        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=num_layers
        )

        # 🔹 Output layer
        self.output_layer = nn.Linear(hidden_dim, 1)


    def forward(self, z_past, x_cov_past, x_cov_future):
        """
        z_past: (B, T, 1)
        x_cov_past: (B, T, d)
        x_cov_future: (B, tau, d)
        """

        B, T, _ = z_past.shape
        tau = x_cov_future.shape[1]

        # 🔹 1. Projection
        p = self.primary_proj(z_past)            # (B, T, H)
        q_past = self.cov_proj(x_cov_past)       # (B, T, H)
        q_future = self.cov_proj(x_cov_future)   # (B, tau, H)

        # 🔹 2. Concatenate + aggregate
        h = torch.cat([p, q_past], dim=-1)       # (B, T, 2H)
        h = self.agg_proj(h)                     # (B, T, H)

        # 🔹 3. Encoder
        memory = self.encoder(h)                 # (B, T, H)

        # 🔹 4. Decoder (forecast)
        tgt = q_future                           # (B, tau, H)
        out_dec = self.decoder(tgt, memory)      # (B, tau, H)

        forecast = self.output_layer(out_dec)    # (B, tau, 1)

        # 🔹 5. Reconstruction (important!)
        recon_dec = self.decoder(q_past, memory)  # (B, T, H)
        recon = self.output_layer(recon_dec)      # (B, T, 1)

        return forecast, recon

###  Compute KMM Weight

In [80]:

# KMM domain adaptiation (Java-papua)
def rbf_kernel(X, Y=None, gamma=None):
    if Y is None:
        Y = X
    if gamma is None:
        gamma = 1.0 / (X.shape[1] * X.var() + 1e-8)
    from sklearn.metrics.pairwise import rbf_kernel as sk_rbf
    return sk_rbf(X, Y, gamma=gamma)

    
def kmm_weights(X_src, X_tgt, B=2.0, eps=0.1, gamma=None):
    """
    Kernel Mean Matching: minimize || (1/n_s) sum_i beta_i phi(x_i) - (1/n_t) sum_j phi(z_j) ||^2
    s.t. 0 <= beta_i <= B, | (1/n_s) sum beta_i - 1 | <= eps.
    """
    n_s, n_t = X_src.shape[0], X_tgt.shape[0]
    K_ss = rbf_kernel(X_src, X_src, gamma=gamma)
    K_st = rbf_kernel(X_src, X_tgt, gamma=gamma)
    kappa = (1.0 / n_t) * K_st.sum(axis=1)
    # min  (1/2) beta' Q beta - kappa' beta   with Q = (1/n_s^2) K_ss
    Q = (1.0 / (n_s * n_s)) * K_ss
    try:
        from cvxopt import matrix, solvers
        solvers.options["show_progress"] = False
        P = matrix(2 * Q)
        q = matrix(-kappa.astype(np.float64))
        G = np.vstack([-np.eye(n_s), np.eye(n_s)])
        h = np.hstack([np.zeros(n_s), B * np.ones(n_s)])
        A = np.ones((1, n_s))
        b = np.array([1.0])
        # | (1/n_s) sum beta - 1 | <= eps  =>  two inequalities
        A2 = np.vstack([A / n_s, -A / n_s])
        b2 = np.array([1 + eps, -(1 - eps)])
        G2 = np.vstack([G, A2])
        h2 = np.hstack([h, b2])
        G2 = matrix(G2.astype(np.float64))
        h2 = matrix(h2.astype(np.float64))
        A = matrix(A.astype(np.float64))
        b = matrix(b.astype(np.float64))
        sol = solvers.qp(P, q, G2, h2, A, b)
        if sol["status"] == "optimal":
            beta = np.array(sol["x"]).ravel().astype(np.float32)
            return np.clip(beta, 0, B)
    except Exception:
        pass
    # Fallback: scipy minimize
    from scipy.optimize import minimize
    def obj(b):
        return 0.5 * b @ Q @ b - kappa @ b

    # | (1/n_s) sum beta - 1 | <= eps  =>  (1-eps) <= sum/n_s <= (1+eps)
    cons = [
        {"type": "ineq", "fun": lambda b: (b.sum() / n_s) - (1 - eps)},
        {"type": "ineq", "fun": lambda b: (1 + eps) - (b.sum() / n_s)},
    ]
    res = minimize(obj, np.ones(n_s), method="SLSQP", bounds=[(0, B)] * n_s, constraints=cons)
    if res.success:
        return np.clip(res.x.astype(np.float32), 0, B)
    return np.ones(n_s, dtype=np.float32)


### Eval Tuning KMM Vu Tran

In [81]:
def eval_kmm_vu_tran_mae(model: torch.nn.Module, X: torch.Tensor, y: torch.Tensor, batch_size: int = 256) -> float:
    if X.shape[0] == 0:
        return float("nan")
    val_abs_sum = 0.0
    val_count = 0
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, X.shape[0], batch_size):
            xb_b = X[i : i + batch_size].to(DEVICE)
            yb_b = y[i : i + batch_size].to(DEVICE)
            z_past = xb_b[:, :INPUT_LEN, :1]
            x_cov_past = xb_b[:, :INPUT_LEN, 1:]
            x_cov_future = xb_b[:, INPUT_LEN:, 1:]

            forecast, _recon = model(z_past, x_cov_past, x_cov_future)
            pred = forecast.squeeze(-1)  # (B, H)

            val_abs_sum += torch.sum(torch.abs(pred - yb_b)).item()
            val_count += pred.numel()
    val_mae = val_abs_sum / max(1, val_count)
    return val_mae


In [82]:
def eval_model_kmm_vu_tran_metrics(model: torch.nn.Module, X: np.ndarray, y: np.ndarray, batch_size):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, X.shape[0], 256):
            xb = torch.from_numpy(X[i : i + 256]).float()
            xb = xb.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))

            z_past = xb[:, :INPUT_LEN, :1]
            x_cov_past = xb[:, :INPUT_LEN, 1:]
            x_cov_future = xb[:, INPUT_LEN:, 1:]

            forecast, _recon = model(z_past, x_cov_past, x_cov_future)
            preds.append(forecast.squeeze(-1).detach().cpu().numpy().astype(np.float32))

    pred = np.concatenate(preds, axis=0)
    y_true = y.astype(np.float32, copy=False)
    err = pred - y_true
    mse = float(np.mean(err ** 2))
    mae = float(np.mean(np.abs(err)))
    rmse = float(np.sqrt(mse))

    kmm_metrics = {"mae": mae, "mse": mse, "rmse": rmse}
    return kmm_metrics

### Exp Train Non-DA

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

def train_transformer_vanilla(
    model,
    X_train, y_train,          # Source (Java) OR Target (Papua) depending on setup
    X_val=None, y_val=None,    # Validation (usually Papua)
    epochs=100,
    batch_size=256,
    lr=1e-3,
    device="cuda"
):
    model.to(device)

    # Convert to tensors
    X_train = torch.from_numpy(X_train).float()
    y_train = torch.from_numpy(y_train).float()

    train_ds = TensorDataset(X_train, y_train)
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    # Standard loss (IMPORTANT: reduction = mean now)
    criterion = nn.MSELoss()

    best_val_mae = float('inf')
    best_state = None
    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            preds = model(xb)

            loss = criterion(preds, yb)   # ✅ no weighting

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

        # Validate on target domain (same as your KMM setup)
    val_mae = eval_kmm_vu_tran_mae(model, X_val, y_val, batch_size=256)

    print(f"Epoch {epoch+1} | Train Loss: {avg_loss:.4f} | Val MAE: {val_mae:.4f}")

        # Early stopping
    if val_mae < best_val_mae:       
        best_val_mae = val_mae
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    return model.to(device), best_val_mae

In [136]:
## Test Non Domain

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

def train_transformer_no_kmm(
    model,
    X_source, y_source,          # Source domain (Java)
    X_val=None, y_val=None,      # Target validation (Papua)
    epochs=20,
    batch_size=256,
    lr=1e-3,
    device="cuda"
):
    """
    Train Transformer with KMM-based domain adaptation.

    X_source: (N, seq_len, features)
    y_source: (N, horizon)
    beta:     (N,) KMM weights
    """

    model.to(device)

    # Convert to tensors
    X_source = torch.from_numpy(X_source).float()
    y_source = torch.from_numpy(y_source).float()
    # beta = torch.from_numpy(beta).float()

    # If provided, convert validation arrays once (we will run the same
    # TemperatureTransformer slicing logic during validation).
    if X_val is not None and y_val is not None:
        X_val = torch.from_numpy(X_val).float()
        y_val = torch.from_numpy(y_val).float()

    # Dataset includes weights
    train_ds = TensorDataset(X_source, y_source)
    train_loader = DataLoader(train_ds, batch_size=256,  num_workers=0)
    # train_loader = DataLoader(train_ds, batch_size=256,  **_dataloader_kwargs(shuffle=True, drop_last=False))

# 
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Important: we need per-sample loss → no reduction
    criterion = nn.MSELoss()

    best_val_mae, best_state = float('inf'), None
    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            z_past = xb[:, :INPUT_LEN, :1]
            x_cov_past = xb[:, :INPUT_LEN, 1:]
            x_cov_future = xb[:, INPUT_LEN:, 1:]

            forecast, _ = model(z_past, x_cov_past, x_cov_future)
            preds = forecast.squeeze(-1)

            # ✅ NORMAL LOSS
            loss = criterion(preds, yb)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        val_mae = eval_kmm_vu_tran_mae(model, X_val, y_val, batch_size)

        print(f"Epoch {epoch+1} | Train Loss: {avg_loss:.4f} | Val MAE: {val_mae:.4f}")

        # ✅ FIXED early stopping
        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        model.load_state_dict(best_state) 
        return model.to(DEVICE), best_val_mae

### EXP Train Domain Adaptation

In [123]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

def train_transformer_weighted_kmm(
    model,
    X_source, y_source,          # Source domain (Java)
    beta,                        # KMM weights (same length as X_source)
    X_val=None, y_val=None,      # Target validation (Papua)
    epochs=20,
    batch_size=256,
    lr=1e-3,
    device="cuda"
):
    """
    Train Transformer with KMM-based domain adaptation.

    X_source: (N, seq_len, features)
    y_source: (N, horizon)
    beta:     (N,) KMM weights
    """

    model.to(device)

    # Convert to tensors
    X_source = torch.from_numpy(X_source).float()
    y_source = torch.from_numpy(y_source).float()
    beta = torch.from_numpy(beta).float()

    # If provided, convert validation arrays once (we will run the same
    # TemperatureTransformer slicing logic during validation).
    if X_val is not None and y_val is not None:
        X_val = torch.from_numpy(X_val).float()
        y_val = torch.from_numpy(y_val).float()

    # Dataset includes weights
    train_ds = TensorDataset(X_source, y_source, beta)
    train_loader = DataLoader(train_ds, batch_size=256,  num_workers=0)
    # train_loader = DataLoader(train_ds, batch_size=256,  **_dataloader_kwargs(shuffle=True, drop_last=False))

# 
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Important: we need per-sample loss → no reduction
    criterion = nn.MSELoss(reduction="none")
    best_val_mae, best_state = float('inf'), None
    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for xb, yb, wb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            wb = wb.to(device)

            optimizer.zero_grad()

            # TemperatureTransformer expects (z_past, x_cov_past, x_cov_future)
            # Here, xb contains stacked past+future features:
            #   xb[:, :INPUT_LEN, :1]   -> z_past
            #   xb[:, :INPUT_LEN, 1:]   -> x_cov_past
            #   xb[:, INPUT_LEN:, 1:]   -> x_cov_future
            z_past = xb[:, :INPUT_LEN, :1]
            x_cov_past = xb[:, :INPUT_LEN, 1:]
            x_cov_future = xb[:, INPUT_LEN:, 1:]

            forecast, _recon = model(z_past, x_cov_past, x_cov_future)
            preds = forecast.squeeze(-1)  # (B, H)

            # Step 1: per-sample loss
            loss_per_sample = criterion(preds, yb)   # (B, H)

            # Step 2: reduce over horizon → (B,)
            loss_per_sample = loss_per_sample.mean(dim=1)

            # Step 3: apply KMM weights
            weighted_loss = (loss_per_sample * wb).mean()

            weighted_loss.backward()
            optimizer.step()

            total_loss += weighted_loss.item()

        avg_loss = total_loss / len(train_loader)
        # Validation on target domain (IMPORTANT for paper alignment)
        # Validation (unweighted): MAE over all elements (N * H)
        # val_abs_sum = 0.0
        # val_count = 0
        # model.eval()
        # with torch.no_grad():
        #     for i in range(0, X_val.shape[0], 256):
        #         xb_b = X_val[i : i + 256].to(device)
        #         yb_b = y_val[i : i + 256].to(device)

        #         z_past = xb_b[:, :INPUT_LEN, :1]
        #         x_cov_past = xb_b[:, :INPUT_LEN, 1:]
        #         x_cov_future = xb_b[:, INPUT_LEN:, 1:]

        #         forecast, _recon = model(z_past, x_cov_past, x_cov_future)
        #         pred = forecast.squeeze(-1)  # (B, H)

        #         val_abs_sum += torch.sum(torch.abs(pred - yb_b)).item()
        #         val_count += pred.numel()

        # val_mae = val_abs_sum / max(1, val_count)
        val_mae = eval_kmm_vu_tran_mae(model, X_val, y_val, batch_size)
        print(f"Epoch {epoch+1} | Train Loss: {avg_loss:.4f} | Val Loss (Target): {val_mae:.4f}")    
        if best_val_mae < val_mae:
            best_val_mae = val_mae
            best_state = {k : v.cpu().clone() for k, v in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    return model.to(DEVICE), best_val_mae

### Run Train KMM

In [ ]:
def run_target_station_experiment_kmm(
    experiment_name: str,
    train_java_df: pd.DataFrame,
    train_papua_df: pd.DataFrame,
    val_papua_df: pd.DataFrame,
    test_papua_df: pd.DataFrame,
    beta: np.ndarray,
    target_station_ids,
    epochs: int = 25,
    batch_size: int = 256,
    lr: float = 1e-3, 
    patience: int = 5,
):
    """
    Train multiple models for a given target-station subset and evaluate on FULL Papua test set:
      - KMM (Java reweighted to selected Papua)
    Optionally includes a precomputed ARIMA baseline dict.
    """
    print("\n" + "=" * 60)
    print(f"{experiment_name}")
    print("=" * 60)

    train_selected_sel = filter_df_by_station_ids(train_papua_df, target_station_ids)
    
    X_src, y_src = build_windows(train_java_df)
    X_tgt, y_tgt = build_windows(train_selected_sel)
    X_tgt_val, y_tgt_val = build_windows(val_papua_df)  # FULL Papua val
    X_tgt_test, y_tgt_test = build_windows(test_papua_df)  # FULL Papua test
    print(f"Java train windows:         {X_src.shape[0]}")
    print(f"Papua train windows (sel):  {X_tgt.shape[0]}")
    print(f"Papua val windows (FULL):   {X_tgt_val.shape[0]}")
    print(f"Papua test windows (FULL):  {X_tgt_test.shape[0]}")
    model_non_da_metrics = {"mae": float("nan"), "mse":float("nan"), "rmse":float("nan")}
    
    X_src_flat = X_src.reshape(X_src.shape[0], -1)
    X_tgt_flat = X_tgt.reshape(X_tgt.shape[0], -1)

    
    subsample_src = min(2000, X_src_flat.shape[0])
    subsample_tgt = min(800, X_tgt_flat.shape[0])
    rs_s = np.random.RandomState(42)
    rs_t = np.random.RandomState(43)
    idx_s = rs_s.choice(X_src_flat.shape[0], subsample_src, replace=False)
    idx_t = rs_t.choice(X_tgt_flat.shape[0], subsample_tgt, replace=False)
            # beta = kmm_weights(X_src_flat[idx_s], X_tgt_flat[idx_t], B=2.0, eps=0.1)
            # Build TemperatureTransformer-compatible windows (past + future covariates).
            # X_*_stack shape: (N, INPUT_LEN + HORIZON, 9) where feature 0 is temperature,
            # and features 1: are the 8 covariates.
    
    z_src_kmm, x_cov_past_src_kmm, x_cov_future_src_kmm, y_src_kmm = build_windows_temp_transformer(train_java_df)
    z_tgt_val_kmm, x_cov_past_tgt_val_kmm, x_cov_future_tgt_val_kmm, y_tgt_val_kmm = build_windows_temp_transformer(val_papua_df)
    z_tgt_test_kmm, x_cov_past_tgt_test_kmm, x_cov_future_tgt_test_kmm, y_tgt_test_kmm = build_windows_temp_transformer(test_papua_df)

    z_tgt_kmm, x_cov_past_tgt_kmm, x_cov_future_tgt_kmm, y_tgt_kmm = build_windows_temp_transformer(train_papua)
    

    def _stack_for_temperature(z_past, x_cov_past, x_cov_future):
        zeros_temp_future = np.zeros((z_past.shape[0], HORIZON, 1), dtype=np.float32)
        past = np.concatenate([z_past, x_cov_past], axis=-1)
        future = np.concatenate([zeros_temp_future, x_cov_future], axis=-1)
        return np.concatenate([past, future], axis=1)

    X_src_stack = _stack_for_temperature(z_src_kmm, x_cov_past_src_kmm, x_cov_future_src_kmm)
    X_tgt_val_stack = _stack_for_temperature(z_tgt_val_kmm, x_cov_past_tgt_val_kmm, x_cov_future_tgt_val_kmm)
    X_tgt_test_stack = _stack_for_temperature(z_tgt_test_kmm, x_cov_past_tgt_test_kmm, x_cov_future_tgt_test_kmm)

    X_tgt_stack = _stack_for_temperature(z_tgt_kmm, x_cov_past_tgt_kmm, x_cov_future_tgt_kmm)

    print("\n" + "-" * 60)
    print("KMM NON Domain-Adaptation")
    print("-" * 60)

    model_non_da = TemperatureTransformer(
        input_dim_primary=1,
        input_dim_cov=8,
        hidden_dim=64,
        num_layers=2,
        forecast_horizon=7
    )

    model_non_da, _ = train_transformer_no_kmm(
        model=model_non_da,
        X_source=X_tgt_stack,
        y_source=y_tgt_kmm,
        X_val=X_tgt_val_stack,
        y_val=y_tgt_val_kmm,
        epochs=100,
        batch_size=256,
        lr=1e-3
    )
    model_non_da_metrics = eval_model_kmm_vu_tran_metrics(model_non_da, X_tgt_test_stack, y_tgt_test_kmm, batch_size=256)
    # -----------------------------
    # C) KMM: reweight Java -> selected Papua
    # -----------------------------
    kmm_metrics = {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")}
    if X_src.shape[0] > 0 and X_tgt.shape[0] > 0:
        print("\n" + "-" * 60)
        print("KMM Domain-Adaptation")
        print("-" * 60)
        try:
            model = TemperatureTransformer(input_dim_primary=1,     # temperature
                    input_dim_cov=8,         # covariates
                    hidden_dim=64,
                    n_heads=4,
                    num_layers=2,
                    forecast_horizon=7)
            model, best_val_loss = train_transformer_weighted_kmm(
                model,
                X_src_stack[idx_s],
                y_src_kmm[idx_s],
                beta,
                X_tgt_val_stack,
                y_tgt_val_kmm,
                epochs=epochs,
                batch_size=batch_size,
                lr=lr,
                device=DEVICE,
            )
            kmm_metrics = eval_model_kmm_vu_tran_metrics(model, X_tgt_test_stack, y_tgt_test_kmm, batch_size)
        except Exception as e:
            print(f"KMM skipped: {e}")
    else:
        print("\n" + "-" * 60)
        print("C) KMM skipped (no Java/source or no target windows)")
        print("-" * 60)

    
    out = {
        "kmm non da": model_non_da_metrics,
        "kmm": kmm_metrics,
    }

    print("\n" + "-" * 60)
    print("Result (Papua test):")
    for k in ["kmm non da","kmm"]:
        m = out[k]
        print(f"  {k:>18s}  MAE={m['mae']:.4f}  MSE={m['mse']:.6f}  RMSE={m['rmse']:.4f}")
    return out


In [137]:
beta = np.load("beta_weights.npy")
res_kmm = run_target_station_experiment_kmm(
        "RUN KMM",
        train_java_df=train_java,
        train_papua_df=train_papua,
        val_papua_df=val_papua,
        test_papua_df=test_papua,
        target_station_ids=[single_id],
        epochs=100,
        batch_size=256,
        lr=1e-3,
        beta=beta,
        patience=5,
)


RUN KMM
Java train windows:         53487
Papua train windows (sel):  2547
Papua val windows (FULL):   8070
Papua test windows (FULL):  8970

------------------------------------------------------------
KMM NON Domain-Adaptation
------------------------------------------------------------
Epoch 1 | Train Loss: 348.2468 | Val MAE: 11.4461

------------------------------------------------------------
KMM Domain-Adaptation
------------------------------------------------------------
Epoch 1 | Train Loss: 629.8771 | Val Loss (Target): 23.3380
Epoch 2 | Train Loss: 570.2875 | Val Loss (Target): 22.7703
Epoch 3 | Train Loss: 544.5621 | Val Loss (Target): 22.1987
Epoch 4 | Train Loss: 517.5992 | Val Loss (Target): 21.6045
Epoch 5 | Train Loss: 489.4770 | Val Loss (Target): 20.9834
Epoch 6 | Train Loss: 461.0185 | Val Loss (Target): 20.3367
Epoch 7 | Train Loss: 432.8415 | Val Loss (Target): 19.6664
Epoch 8 | Train Loss: 404.7730 | Val Loss (Target): 18.9746
Epoch 9 | Train Loss: 376.8516 | V

KeyboardInterrupt: 

## ATTF

In [58]:
class ValueEmbedding(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

    def forward(self, x):
        # x: (B, T, F)
        return self.mlp(x)  # (B, T, H)

In [59]:
class PatternEmbedding(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.conv = nn.Conv1d(
            in_channels=input_dim,
            out_channels=hidden_dim,
            kernel_size=3,
            padding=1
        )

    def forward(self, x):
        # x: (B, T, F)
        x = x.transpose(1, 2)  # (B, F, T)
        p = self.conv(x)       # (B, H, T)
        return p.transpose(1, 2)  # (B, T, H)

In [60]:
class PrivateEncoder(nn.Module):
    def __init__(self, input_dim, d_model):
        super().__init__()
        self.value_emb = ValueEmbedding(input_dim, d_model)
        self.pattern_emb = PatternEmbedding(input_dim, d_model)

    def forward(self, x):
        v = self.value_emb(x)   # values (domain-specific)
        p = self.pattern_emb(x) # patterns (used for Q,K)
        return v, p

In [61]:
class SharedAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, pattern, value):
        # pattern: (B, T, H)
        # value:   (B, T, H)

        Q = self.q_proj(pattern)   # (B, T, H)
        K = self.k_proj(pattern)   # (B, T, H)

        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / (pattern.size(-1) ** 0.5)
        attn_weights = torch.softmax(attn_scores, dim=-1)

        # IMPORTANT: V = value (NOT pattern)
        out = torch.matmul(attn_weights, value)  # (B, T, H)

        return out

In [62]:
class Decoder(nn.Module):
    def __init__(self, hidden_dim, output_steps):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_steps)
        )

    def forward(self, x):
        # x: (B, T, H)
        x = x[:, -1, :]  # last timestep
        return self.fc(x)  # (B, 7)

In [77]:
class AttF(nn.Module):
    def __init__(self, input_dim=9, hidden_dim=64, output_steps=7):
        super().__init__()

        self.value_embed = ValueEmbedding(input_dim, hidden_dim)
        self.pattern_embed = PatternEmbedding(input_dim, hidden_dim)
        self.attention = SharedAttention(hidden_dim)
        self.decoder = Decoder(hidden_dim, output_steps)
        self.reconstruction_head = nn.Linear(hidden_dim, input_dim)

    def forward(self, x):
        v = self.value_embed(x)
        p = self.pattern_embed(x)
        attn_out = self.attention(p, v)

        y_hat = self.decoder(attn_out)         # (B, 7)
        x_recon = self.reconstruction_head(attn_out)
        
        return y_hat, x_recon

## DAF 

In [90]:
class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_ * grad_output, None


def grad_reverse(x, lambda_=1.0):
    return GradReverse.apply(x, lambda_)

In [92]:
class DomainDiscriminator(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        return self.net(x)

In [93]:
class DAF(nn.Module):
    def __init__(self, input_dim=9, hidden_dim=64, output_steps=7):
        super().__init__()

        self.value_embed = ValueEmbedding(input_dim, hidden_dim)
        self.pattern_embed = PatternEmbedding(input_dim, hidden_dim)
        self.attention = SharedAttention(hidden_dim)
        self.decoder = Decoder(hidden_dim, output_steps)
        self.reconstruction_head = nn.Linear(hidden_dim, input_dim)

    def forward(self, x):
        v = self.value_embed(x)
        p = self.pattern_embed(x)          # 🔥 THIS is what we align

        attn_out = self.attention(p, v)

        y_hat = self.decoder(attn_out)
        x_recon = self.reconstruction_head(attn_out)

        return y_hat, x_recon, p

### Train ATTF early stop MAE

In [83]:
def attf_eval_model_mae(model: torch.nn.Module, X: np.ndarray, y: np.ndarray, batch_size: int = 256) -> float:
    if X.shape[0] == 0:
        return float("nan")
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, X.shape[0], batch_size):
            xb = torch.from_numpy(X[i : i + batch_size])
            xb = xb.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
            pred, _ = model(xb)
            # pred = pred[:, -1, :]
            # pred = pred[:, :y.shape[1], 0]
            preds.append(pred.detach().cpu().numpy().astype(np.float32))
    pred = np.concatenate(preds, axis=0)
    return float(np.mean(np.abs(pred - y)))

In [84]:
def attf_eval_model_metrics(model: torch.nn.Module, X: np.ndarray, y: np.ndarray, batch_size: int = 256):
    """
    Compute MAE/MSE/RMSE on window targets (y shape: [N, H]).
    Metrics are computed over all elements (N*H).
    Returns dict: {"mae": float, "mse": float, "rmse": float}.
    """
    if X.shape[0] == 0:
        return {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")}
    model.eval() # switch model into evaluation mode
    preds = []
    with torch.no_grad(): # disable gradients
        for i in range(0, X.shape[0], batch_size):
            # Extract batch 
            xb = torch.from_numpy(X[i : i + batch_size])
            xb = xb.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
            pred, _ = model(xb)            
            preds.append(pred.detach().cpu().numpy().astype(np.float32))
    pred = np.concatenate(preds, axis=0).astype(np.float32)
    y = y.astype(np.float32, copy=False)
    err = (pred - y).astype(np.float32)
    mse = float(np.mean(err ** 2))
    mae = float(np.mean(np.abs(err)))
    rmse = float(np.sqrt(mse))
    return {"mae": mae, "mse": mse, "rmse": rmse}


In [ ]:
def train_attf_earlystop_target_mae(
    model: nn.Module,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_tgt_val: np.ndarray,
    y_tgt_val: np.ndarray,
    epochs: int = 100,
    batch_size: int = 256,
    lr: float = 1e-3,
    patience: int = 5,
    lambda_recon: float = 1e-4,  # 🔥 important
):
    """
    ATTF Training:
    - Forecast loss (MSE)
    - Reconstruction loss (MSE)
    - Early stop on target MAE
    """
    from torch.utils.data import TensorDataset, DataLoader

    if X_train.shape[0] == 0:
        return model.to(DEVICE), float("nan")

    model = model.to(DEVICE).float()

    train_ds = TensorDataset(
        torch.from_numpy(X_train).to(DEVICE),
        torch.from_numpy(y_train).to(DEVICE),
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    opt = torch.optim.Adam(model.parameters(), lr=lr)

    best_mae, best_state, wait = float("inf"), None, 0

    for ep in range(epochs):
        model.train()
        tr_loss = 0.0

        for xb, yb in train_loader:
            opt.zero_grad()

            # 🔥 ATTF outputs
            y_pred, x_recon = model(xb)

            # 🔹 Forecast loss
            loss_forecast = nn.functional.mse_loss(y_pred, yb)

            # 🔹 Reconstruction loss
            # loss_recon = nn.functional.mse_loss(x_recon, xb)
            loss_recon = ((x_recon - xb) ** 2).mean(dim=(1,2)).mean()

            # 🔥 Total loss (paper)
            loss = loss_forecast + lambda_recon * loss_recon

            loss.backward()
            opt.step()

            tr_loss += loss.item() * xb.size(0)

        tr_loss /= max(1, X_train.shape[0])

        # 🔹 Evaluate on target domain (unchanged)
        val_mae = attf_eval_model_mae(model, X_tgt_val, y_tgt_val, batch_size=256)
        print(f"Loss Foreacst: {loss_forecast.item()} | Loss Recon: {loss_recon.item()}")
        print(
            f"  Epoch {ep+1}/{epochs}  "
            f"Train Loss: {tr_loss:.6f}  "
            f"Target Val MAE: {val_mae:.4f}"
        )

        # 🔹 Early stopping (same logic)
        if val_mae < best_mae:
            best_mae = val_mae
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            # wait = 0
        # else:
        #     wait += 1
        #     if wait >= patience:
        #         print(f"  Early stopping at epoch {ep+1}")
        #         break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model.to(DEVICE), best_mae

### train daf early stop mae

In [ ]:
### Train ATTF early stop MAE
def daf_eval_model_mae(model: torch.nn.Module, X: np.ndarray, y: np.ndarray, batch_size: int = 256) -> float:
    if X.shape[0] == 0:
        return float("nan")
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, X.shape[0], batch_size):
            xb = torch.from_numpy(X[i : i + batch_size])
            xb = xb.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
            pred, _, _ = model(xb)
            # pred = pred[:, -1, :]
            # pred = pred[:, :y.shape[1], 0]
            preds.append(pred.detach().cpu().numpy().astype(np.float32))
    pred = np.concatenate(preds, axis=0)
    return float(np.mean(np.abs(pred - y)))


In [96]:
def daf_eval_model_metrics(model: torch.nn.Module, X: np.ndarray, y: np.ndarray, batch_size: int = 256):
    """
    Compute MAE/MSE/RMSE on window targets (y shape: [N, H]).
    Metrics are computed over all elements (N*H).
    Returns dict: {"mae": float, "mse": float, "rmse": float}.
    """
    if X.shape[0] == 0:
        return {"mae": float("nan"), "mse": float("nan"), "rmse": float("nan")}
    model.eval() # switch model into evaluation mode
    preds = []
    with torch.no_grad(): # disable gradients
        for i in range(0, X.shape[0], batch_size):
            # Extract batch 
            xb = torch.from_numpy(X[i : i + batch_size])
            xb = xb.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
            pred, _, _ = model(xb)            
            preds.append(pred.detach().cpu().numpy().astype(np.float32))
    pred = np.concatenate(preds, axis=0).astype(np.float32)
    y = y.astype(np.float32, copy=False)
    err = (pred - y).astype(np.float32)
    mse = float(np.mean(err ** 2))
    mae = float(np.mean(np.abs(err)))
    rmse = float(np.sqrt(mse))
    return {"mae": mae, "mse": mse, "rmse": rmse}


In [110]:
def train_daf_earlystop_target_mae(
    model: nn.Module,
    discriminator: nn.Module,
    X_src: np.ndarray,
    y_src: np.ndarray,
    X_tgt: np.ndarray,
    X_tgt_val: np.ndarray,
    y_tgt_val: np.ndarray,
    epochs: int = 100,
    batch_size: int = 256,
    lr: float = 1e-3,
    lambda_recon: float = 0.5,
    lambda_domain: float = 0.1,
):
    """
    DAF Training:
    - Source supervised (forecast)
    - Target unsupervised (reconstruction)
    - Domain adversarial alignment (pattern space)
    - Early stop on target MAE
    """

    from torch.utils.data import DataLoader, TensorDataset

    model = model.to(DEVICE).float()
    discriminator = discriminator.to(DEVICE).float()

    # 🔹 Datasets
    src_ds = TensorDataset(
        torch.from_numpy(X_src).to(DEVICE),
        torch.from_numpy(y_src).to(DEVICE)
    )
    tgt_ds = TensorDataset(
        torch.from_numpy(X_tgt).to(DEVICE)
    )

    src_loader = DataLoader(src_ds, batch_size=batch_size, shuffle=True, drop_last=True)
    tgt_loader = DataLoader(tgt_ds, batch_size=batch_size, shuffle=True, drop_last=True)

    opt_model = torch.optim.Adam(model.parameters(), lr=lr)
    opt_disc = torch.optim.Adam(discriminator.parameters(), lr=lr)

    bce = nn.BCEWithLogitsLoss()

    best_mae, best_state = float("inf"), None

    for ep in range(epochs):
        model.train()
        discriminator.train()

        tgt_iter = iter(tgt_loader)

        total_loss = 0.0

        for xb_src, yb_src in src_loader:
            try:
                xb_tgt = next(tgt_iter)[0]
            except StopIteration:
                tgt_iter = iter(tgt_loader)
                xb_tgt = next(tgt_iter)[0]

            # =========================
            # 🔷 1. Forward pass
            # =========================
            y_pred_src, x_recon_src, p_src = model(xb_src)
            _, x_recon_tgt, p_tgt = model(xb_tgt)

            # =========================
            # 🔷 2. Forecast loss (source only)
            # =========================
            loss_forecast = nn.functional.mse_loss(y_pred_src, yb_src)

            # =========================
            # 🔷 3. Reconstruction loss (both domains)
            # =========================
            loss_recon_src = nn.functional.mse_loss(x_recon_src, xb_src)
            loss_recon_tgt = nn.functional.mse_loss(x_recon_tgt, xb_tgt)

            loss_recon = (loss_recon_src + loss_recon_tgt) / 2

            # =========================
            # 🔷 4. Domain loss (pattern features)
            # =========================
            # Pool temporal dimension
            p_src_mean = torch.mean(p_src, dim=1)
            p_tgt_mean = torch.mean(p_tgt, dim=1)

            # Labels
            domain_src = torch.zeros(p_src_mean.size(0), 1).to(DEVICE)
            domain_tgt = torch.ones(p_tgt_mean.size(0), 1).to(DEVICE)

            # 🔹 Train discriminator
            d_src = discriminator(p_src_mean.detach())
            d_tgt = discriminator(p_tgt_mean.detach())

            loss_disc = bce(d_src, domain_src) + bce(d_tgt, domain_tgt)

            opt_disc.zero_grad()
            loss_disc.backward()
            opt_disc.step()

            # 🔹 Train generator (adversarial)
            p_src_rev = grad_reverse(p_src_mean)
            p_tgt_rev = grad_reverse(p_tgt_mean)

            d_src = discriminator(p_src_rev)
            d_tgt = discriminator(p_tgt_rev)

            loss_domain = bce(d_src, domain_tgt) + bce(d_tgt, domain_src)

            # =========================
            # 🔥 5. Total loss
            # =========================
            loss = (
                loss_forecast
                + lambda_recon * loss_recon
                + lambda_domain * loss_domain
            )

            opt_model.zero_grad()
            loss.backward()
            opt_model.step()

            total_loss += loss.item()

        # =========================
        # 🔷 6. Validation
        # =========================
        val_mae = daf_eval_model_mae(model, X_tgt_val, y_tgt_val)

        print(
            f"Epoch {ep+1}/{epochs} | "
            f"Loss: {total_loss:.4f} | "
            f"Val MAE: {val_mae:.4f}"
        )

        if val_mae < best_mae:
            best_mae = val_mae
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    return model.to(DEVICE), best_mae

In [111]:
def run_attf_daf_target_station_experiment(
    experiment_name: str,
    train_java_df: pd.DataFrame,
    train_papua_df: pd.DataFrame,
    val_papua_df: pd.DataFrame,
    test_papua_df: pd.DataFrame,
    target_station_ids,
    epochs: int = 25,
    batch_size: int = 64,
    lr: float = 1e-3,
    patience: int = 5,
):
    """
    Train multiple models for a given target-station subset and evaluate on FULL Papua test set:
      - Vanilla Transformer (Papua-only, supervised)
      - DANN (Java vs selected Papua)
      - KMM (Java reweighted to selected Papua)
      - Climate-Aware Transformer (Papua-only, supervised)
      - Climate-Aware + DANN (Java vs selected Papua)
    Optionally includes a precomputed ARIMA baseline dict.
    """
    print("\n" + "=" * 60)
    print(f"{experiment_name}")
    print("=" * 60)
    print(f"Selected Papua target stations (train only): {', '.join([str(x) for x in target_station_ids])}")

    train_papua_sel = filter_df_by_station_ids(train_papua_df, target_station_ids)

    X_src, y_src = build_windows(train_java_df)
    X_tgt, y_tgt = build_windows(train_papua_sel)
    X_tgt_val, y_tgt_val = build_windows(val_papua_df)  # FULL Papua val
    X_tgt_test, y_tgt_test = build_windows(test_papua_df)  # FULL Papua test

    print(f"Java train windows:         {X_src.shape[0]}")
    print(f"Papua train windows (sel):  {X_tgt.shape[0]}")
    print(f"Papua val windows (FULL):   {X_tgt_val.shape[0]}")
    print(f"Papua test windows (FULL):  {X_tgt_test.shape[0]}")
    

    print("\n" + "-" * 60)
    print("Attention Sharing Non-DA")
    print("-" * 60)
    attf = AttF(
        input_dim=X_tgt.shape[2],
        hidden_dim=64,
        output_steps=7,
    )
    attf, best_val_mae = train_attf_earlystop_target_mae(
        attf,
        X_tgt,
        y_tgt,
        X_tgt_val,
        y_tgt_val,
        epochs=epochs,
        batch_size=batch_size,
        lr=lr,
        patience=patience,
    )
    attf_metrics = attf_eval_model_metrics(attf, X_tgt_test, y_tgt_test, batch_size=256)


    print(f"Best Papua Val MAE (early stop): {best_val_mae:.4f}")
    print(
        f"Papua Test  MAE: {attf_metrics['mae']:.4f}  "
        f"RMSE: {attf_metrics['rmse']:.4f}  MSE: {attf_metrics['mse']:.6f}"
    )

    
    print("\n" + "-" * 60)
    print("Attention Sharing Domain adaptation")
    print("-" * 60)

    print(X_tgt.shape)
    daf = DAF(
        input_dim=X_tgt.shape[2],
        hidden_dim=64,
        output_steps=7,
    )
    discriminator = DomainDiscriminator(hidden_dim=64)
    daf, best_val_mae = train_daf_earlystop_target_mae(
        model=daf,
        discriminator=discriminator,
        X_src=X_src,
        y_src=y_src,
        X_tgt=X_tgt,
        X_tgt_val=X_tgt_val,
        y_tgt_val=y_tgt_val,
        epochs=100,
        batch_size=256,
        lr=1e-3,
        lambda_recon=0.5,
        lambda_domain=0.1   # IMPORTANT hyperparameter
    )
    daf_metrics = daf_eval_model_metrics(daf, X_tgt_test, y_tgt_test, batch_size=256)


    print(f"Best Papua Val MAE (early stop): {best_val_mae:.4f}")
    print(
        f"Papua Test  MAE: {daf_metrics['mae']:.4f}  "
        f"RMSE: {daf_metrics['rmse']:.4f}  MSE: {daf_metrics['mse']:.6f}"
    )


    out = {
        "attf": attf_metrics,
        "daf": daf_metrics,
        
    }

    print("\n" + "-" * 60)
    print("Result (Papua test):")
    for k in ["attf", "daf"]:
        m = out[k]
        print(f"  {k:>18s}  MAE={m['mae']:.4f}  MSE={m['mse']:.6f}  RMSE={m['rmse']:.4f}")
    return out



In [112]:
run_attf_daf_target_station_experiment(
    "RUN ATTF & DAF",
    train_java_df=train_java,
    train_papua_df=train_papua,
    val_papua_df=val_papua,
    test_papua_df=test_papua,
    target_station_ids=[single_id],
    epochs=100,
    batch_size=256,
    lr=1e-3,
    patience=5
)


RUN ATTF & DAF
Selected Papua target stations (train only): P11
Java train windows:         53487
Papua train windows (sel):  2547
Papua val windows (FULL):   8070
Papua test windows (FULL):  8970

------------------------------------------------------------
Attention Sharing Non-DA
------------------------------------------------------------
  Epoch 1/100  Train Loss: 2237.738909  Target Val MAE: 17.8170
  Epoch 2/100  Train Loss: 638.107417  Target Val MAE: 37.3806
  Epoch 3/100  Train Loss: 112.164914  Target Val MAE: 25.2780
  Epoch 4/100  Train Loss: 63.749461  Target Val MAE: 26.1645
  Epoch 5/100  Train Loss: 29.681347  Target Val MAE: 28.0773
  Epoch 6/100  Train Loss: 20.361376  Target Val MAE: 27.1119
  Epoch 7/100  Train Loss: 17.501291  Target Val MAE: 27.7176
  Epoch 8/100  Train Loss: 15.022572  Target Val MAE: 27.1322
  Epoch 9/100  Train Loss: 13.955538  Target Val MAE: 27.4811
  Epoch 10/100  Train Loss: 12.844885  Target Val MAE: 27.2819
  Epoch 11/100  Train Loss: 1

{'attf': {'mae': 17.882604598999023,
  'mse': 458.4377746582031,
  'rmse': 21.411160049334157},
 'daf': {'mae': 1.076913595199585,
  'mse': 1.9339165687561035,
  'rmse': 1.3906532884785134}}

### Exp improve ATTF

In [136]:
class PrivateEncoder(nn.Module):
    """
    Produces:
      P  [B, T, d_model]  – multi-scale pattern embedding (concat of M convs)
      V  [B, T, d_model]  – value embedding (position-wise MLP on raw input)

    Args:
        input_dim   : number of input features per time step (1 for univariate)
        d_model     : output dimension for both P and V
        kernel_sizes: tuple of conv kernel sizes, one conv branch per entry (M branches)
        n_mlp_layers: depth of the value MLP
    """

    def __init__(
        self,
        input_dim: int = 1,
        d_model: int = 64,
        kernel_sizes: tuple = (3, 5),
        n_mlp_layers: int = 2,
    ):
        super().__init__()
        self.M = len(kernel_sizes)
        self.d_model = d_model

        # --- Value MLP  (position-wise, shared across time) ---
        value_layers = [nn.Linear(input_dim, d_model), nn.ReLU()]
        for _ in range(n_mlp_layers - 1):
            value_layers += [nn.Linear(d_model, d_model), nn.ReLU()]
        self.value_mlp = nn.Sequential(*value_layers)

        # --- Pattern convolutions  (M independent temporal convs) ---
        # Each conv maps input_dim → (d_model // M) channels so that
        # concatenation across M branches gives exactly d_model channels.
        branch_dim = d_model // self.M
        self.pattern_convs = nn.ModuleList()
        for ks in kernel_sizes:
            pad = ks // 2           # "same" padding to preserve length
            self.pattern_convs.append(
                nn.Sequential(
                    # Conv1d expects (B, C_in, T)
                    nn.Conv1d(input_dim, branch_dim, kernel_size=ks, padding=pad),
                    nn.ReLU(),
                )
            )
        # If d_model isn't evenly divisible by M, a projection fixes the dim
        conv_out_dim = branch_dim * self.M
        self.pattern_proj = (
            nn.Linear(conv_out_dim, d_model)
            if conv_out_dim != d_model
            else nn.Identity()
        )

    def forward(self, x: torch.Tensor):
        """
        x : [B, T, input_dim]
        returns P [B, T, d_model], V [B, T, d_model]
        """
        # Value embedding
        V = self.value_mlp(x)                           # [B, T, d_model]

        # Pattern embedding: run each conv branch on (B, input_dim, T)
        x_t = x.permute(0, 2, 1)                        # [B, input_dim, T]
        branches = [conv(x_t).permute(0, 2, 1)          # [B, T, branch_dim]
                    for conv in self.pattern_convs]
        P = torch.cat(branches, dim=-1)                  # [B, T, conv_out_dim]
        P = self.pattern_proj(P)                         # [B, T, d_model]

        return P, V


In [137]:
import math

class AttentionModule(nn.Module):
    """
    Shared attention module (Section 4.1).

    Computes Q, K from pattern embeddings P via a position-wise MLP,
    then produces context vectors O via scaled-dot-product attention
    on value embeddings V.

    Two operational modes:
      - interpolation (t ≤ T) : reconstruct input  – each position attends to
                                 all *other* positions (leave-one-out mask).
      - extrapolation (t > T) : forecast future     – each future query attends
                                 to a causal window of historical keys/values.

    Args:
        d_model    : query/key/value dimension
        n_mlp_layers: depth of the Q/K projection MLP
        kernel_size: convolution kernel size s used in the encoder
                     (needed to derive the neighbourhood offset s̄ = ⌈(s-1)/2⌉)
    """

    def __init__(self, d_model: int = 64, n_mlp_layers: int = 2, kernel_size: int = 3):
        super().__init__()
        self.d_model = d_model
        self.scale = math.sqrt(d_model)
        # s̄ = ⌈(s-1)/2⌉  used in extrapolation (eq. after Eq.6 in paper)
        self.s_bar = math.ceil((kernel_size - 1) / 2)

        # Position-wise MLP  P → (Q, K),  output dim = 2 * d_model
        qk_layers = [nn.Linear(d_model, d_model), nn.ReLU()]
        for _ in range(n_mlp_layers - 1):
            qk_layers += [nn.Linear(d_model, d_model), nn.ReLU()]
        qk_layers.append(nn.Linear(d_model, 2 * d_model))
        self.qk_mlp = nn.Sequential(*qk_layers)

        # Output projection MLP  (Eq.6)
        self.out_mlp = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model),
        )

    def _scaled_dot(self, q: torch.Tensor, k: torch.Tensor) -> torch.Tensor:
        """Scaled dot-product: q [B, Nq, d], k [B, Nk, d] → scores [B, Nq, Nk]"""
        return torch.bmm(q, k.transpose(1, 2)) / self.scale

    # ------------------------------------------------------------------ #
    #  Interpolation: attend to all positions except self (leave-one-out)
    # ------------------------------------------------------------------ #
    def _interpolate(self, Q, K, V):
        """
        Q, K, V : [B, T, d_model]
        Returns  : [B, T, d_model]  – reconstructed context
        """
        B, T, _ = Q.shape
        scores = self._scaled_dot(Q, K)          # [B, T, T]

        # Mask out diagonal so position t never attends to itself
        mask = torch.eye(T, device=Q.device, dtype=torch.bool)
        scores = scores.masked_fill(mask.unsqueeze(0), float("-inf"))

        weights = F.softmax(scores, dim=-1)       # [B, T, T]
        context = torch.bmm(weights, V)           # [B, T, d_model]
        return context

    # ------------------------------------------------------------------ #
    #  Extrapolation: future query attends to a causal window
    # ------------------------------------------------------------------ #
    def _extrapolate_one(self, q_t: torch.Tensor, K: torch.Tensor, V: torch.Tensor,
                          s: int, T: int):
        """
        Generate one future step using the extrapolation rule (Section 4.1).

        q_t  : [B, d_model]  query for the future step  (= q_{T - s̄})
        K    : [B, T, d_model]
        V    : [B, T, d_model]

        Neighbourhood of keys: N(T+1) = {s, …, T - s̄ - 1}   (0-indexed here)
        Paired value index   : µ(t') = t' + s̄ + 1
        """
        s_bar = self.s_bar
        # key indices: [s-1 … T - s_bar - 2]  (0-indexed; paper uses 1-indexed)
        key_start = max(s - 1, 0)
        key_end   = max(T - s_bar - 1, key_start + 1)   # at least 1 key
        k_slice   = K[:, key_start:key_end, :]           # [B, Nk, d]

        # Corresponding value indices: t' + s_bar + 1
        val_indices = [min(i + s_bar + 1, T - 1)
                       for i in range(key_start, key_end)]
        v_slice = V[:, val_indices, :]                   # [B, Nk, d]

        scores  = torch.bmm(q_t.unsqueeze(1), k_slice.transpose(1, 2)) / self.scale
        weights = F.softmax(scores, dim=-1)              # [B, 1, Nk]
        context = torch.bmm(weights, v_slice).squeeze(1) # [B, d_model]
        return context

    # ------------------------------------------------------------------ #
    #  Forward
    # ------------------------------------------------------------------ #
    def forward(self, P: torch.Tensor, V: torch.Tensor,
                future_steps: int = 0, kernel_size: int = None):
        """
        P            : [B, T, d_model]  pattern embeddings from encoder
        V            : [B, T, d_model]  value embeddings from encoder
        future_steps : τ  number of autoregressive steps to forecast
        kernel_size  : override the module-level kernel_size for s̄ computation

        Returns:
            O_recon  [B, T, d_model]  – interpolation context (reconstruction)
            O_fore   [B, τ, d_model]  – extrapolation context (forecast)
                                         empty tensor when future_steps == 0
        """
        B, T, _ = P.shape
        s = kernel_size or (2 * self.s_bar + 1)   # recover s from s̄

        # Project P → Q, K
        qk = self.qk_mlp(P)                       # [B, T, 2*d_model]
        Q, K = qk.chunk(2, dim=-1)                # each [B, T, d_model]

        # --- Interpolation (reconstruction) ---
        O_recon_raw = self._interpolate(Q, K, V)
        O_recon = self.out_mlp(O_recon_raw)        # [B, T, d_model]

        # --- Extrapolation (forecast) ---
        if future_steps == 0:
            O_fore = torch.zeros(B, 0, self.d_model, device=P.device)
            return O_recon, O_fore, Q, K

        # The extrapolation query for step T+1 is q_{T - s̄}  (paper eq.)
        q_extra = Q[:, T - self.s_bar - 1, :]     # [B, d_model]

        fore_contexts = []
        # We build the context for each future step in an autoregressive
        # fashion; since AttF doesn't update K/V with future values in the
        # attention (it feeds predictions back through the encoder), we
        # keep K and V fixed from the historical window for the attention.
        for _ in range(future_steps):
            ctx = self._extrapolate_one(q_extra, K, V, s, T)
            ctx = self.out_mlp(ctx)                # [B, d_model]
            fore_contexts.append(ctx.unsqueeze(1)) # [B, 1, d_model]

        O_fore = torch.cat(fore_contexts, dim=1)   # [B, τ, d_model]
        return O_recon, O_fore, Q, K


In [138]:
class PrivateDecoder(nn.Module):
    """
    Position-wise MLP: context o_t → scalar prediction z_hat_t  (Eq.6 / §4.1)

    Args:
        d_model    : input dimension
        output_dim : prediction dimension (1 for univariate)
        n_mlp_layers: depth
    """

    def __init__(self, d_model: int = 64, output_dim: int = 1, n_mlp_layers: int = 2):
        super().__init__()
        layers = [nn.Linear(d_model, d_model), nn.ReLU()]
        for _ in range(n_mlp_layers - 1):
            layers += [nn.Linear(d_model, d_model), nn.ReLU()]
        layers.append(nn.Linear(d_model, output_dim))
        self.mlp = nn.Sequential(*layers)

    def forward(self, o: torch.Tensor) -> torch.Tensor:
        """o : [B, T_or_τ, d_model] → [B, T_or_τ, output_dim]"""
        return self.mlp(o)


In [144]:
def run_attf_daf_target_station_experiment(
    experiment_name: str,
    train_java_df: pd.DataFrame,
    train_papua_df: pd.DataFrame,
    val_papua_df: pd.DataFrame,
    test_papua_df: pd.DataFrame,
    target_station_ids,
    epochs: int = 25,
    batch_size: int = 64,
    lr: float = 1e-3,
    patience: int = 5,
):
    """
    Train multiple models for a given target-station subset and evaluate on FULL Papua test set:
      - Vanilla Transformer (Papua-only, supervised)
      - DANN (Java vs selected Papua)
      - KMM (Java reweighted to selected Papua)
      - Climate-Aware Transformer (Papua-only, supervised)
      - Climate-Aware + DANN (Java vs selected Papua)
    Optionally includes a precomputed ARIMA baseline dict.
    """
    print("\n" + "=" * 60)
    print(f"{experiment_name}")
    print("=" * 60)
    print(f"Selected Papua target stations (train only): {', '.join([str(x) for x in target_station_ids])}")

    train_papua_sel = filter_df_by_station_ids(train_papua_df, target_station_ids)

    X_src, y_src = build_windows(train_java_df)
    X_tgt, y_tgt = build_windows(train_papua_sel)
    X_tgt_val, y_tgt_val = build_windows(val_papua_df)  # FULL Papua val
    X_tgt_test, y_tgt_test = build_windows(test_papua_df)  # FULL Papua test

    print(f"Java train windows:         {X_src.shape[0]}")
    print(f"Papua train windows (sel):  {X_tgt.shape[0]}")
    print(f"Papua val windows (FULL):   {X_tgt_val.shape[0]}")
    print(f"Papua test windows (FULL):  {X_tgt_test.shape[0]}")
    

    print("\n" + "-" * 60)
    print("Attention Sharing Non-DA")
    print("-" * 60)
    attf = AttF(
        input_dim=9,
        output_dim= 1,
        d_model= 64,
        kernel_sizes= (3, 5),
        n_enc_layers = 2,
        n_attn_layers = 2,
        n_dec_layers = 2

    )
    attf, best_val_mae = train_attf(
        attf,
        X_tgt,
        y_tgt,
        X_tgt_val,
        y_tgt_val,
        tau=7,
        epochs=epochs,
        batch_size=batch_size,
        lr=lr,
        patience=patience,
        lambda_recon = 0.0001
    )
    attf_metrics = attf_eval_model_metrics(attf, X_tgt_test, y_tgt_test, batch_size=256)


    print(f"Best Papua Val MAE (early stop): {best_val_mae:.4f}")
    print(
        f"Papua Test  MAE: {attf_metrics['mae']:.4f}  "
        f"RMSE: {attf_metrics['rmse']:.4f}  MSE: {attf_metrics['mse']:.6f}"
    )


    return attf_metrics

